In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Gold Data Warehouse — Step 01: Define Gold Analytical Model
#
# Purpose:
# Define the business-oriented analytical model that will be
# constructed from validated Silver Delta tables.
#
# Gold design principles:
# 1. Use only fields and relationships supported by the source.
# 2. Preserve validated Silver business meaning.
# 3. Organize data for SQL, Power BI, and Python analytics.
# 4. Avoid unnecessary copies of the 21 Silver source tables.
# 5. Establish explicit grain and lineage before persistence.
#
# No Gold tables are created in this step.

print("GOLD DATA WAREHOUSE — ANALYTICAL MODEL DESIGN")
print("=" * 120)

silver_catalog = "workspace"
silver_schema = "silver_sop"
gold_catalog = "workspace"
gold_schema = "gold_sop"


# ----------------------------------------------------------------
# 1. Proposed Gold analytical model
# ----------------------------------------------------------------

gold_model = [

    {
        "gold_table": "dim_product",
        "table_type": "DIMENSION",
        "grain": "One row per product",
        "primary_sources": [
            "finished_product_type",
            "finished_product_economics",
            "product_inventory",
            "product_lot_size"
        ],
        "analytical_purpose":
            "Central product attributes, economics, inventory, "
            "and lot-size context."
    },

    {
        "gold_table": "dim_planning_period",
        "table_type": "DIMENSION",
        "grain": "One row per planning period",
        "primary_sources": [
            "planning_periods"
        ],
        "analytical_purpose":
            "Planning-horizon period structure for demand "
            "and capacity analysis."
    },

    {
        "gold_table": "dim_machine",
        "table_type": "DIMENSION",
        "grain": "One row per production machine",
        "primary_sources": [
            "machine"
        ],
        "analytical_purpose":
            "Production-machine reference and unit-time cost."
    },

    {
        "gold_table": "dim_packing_machine",
        "table_type": "DIMENSION",
        "grain": "One row per packing machine",
        "primary_sources": [
            "packing_machine"
        ],
        "analytical_purpose":
            "Packing-machine reference and unit-time cost."
    },

    {
        "gold_table": "fact_customer_order",
        "table_type": "FACT",
        "grain": "One row per customer order",
        "primary_sources": [
            "customer_order"
        ],
        "analytical_purpose":
            "Order quantity, sales value, due-date position, "
            "and planning-horizon status analysis."
    },

    {
        "gold_table": "fact_demand_plan",
        "table_type": "FACT",
        "grain": "One row per finished product per planning period",
        "primary_sources": [
            "demand_plan"
        ],
        "analytical_purpose":
            "Adjusted demand and target-stock analysis "
            "across the planning horizon."
    },

    {
        "gold_table": "fact_machine_capacity",
        "table_type": "FACT",
        "grain": "One row per production machine per planning period",
        "primary_sources": [
            "machine_capacity"
        ],
        "analytical_purpose":
            "Regular-time and overtime production-capacity analysis."
    },

    {
        "gold_table": "fact_packing_capacity",
        "table_type": "FACT",
        "grain": "One row per packing machine per planning period",
        "primary_sources": [
            "packing_capacity"
        ],
        "analytical_purpose":
            "Regular-time and overtime packing-capacity analysis."
    },

    {
        "gold_table": "bridge_product_production",
        "table_type": "BRIDGE",
        "grain": "One row per product-production alternative",
        "primary_sources": [
            "product_production_alternative"
        ],
        "analytical_purpose":
            "Connect products to supported production alternatives "
            "while preserving source main-alternative codes."
    },

    {
        "gold_table": "bridge_alternative_machine",
        "table_type": "BRIDGE",
        "grain": "One row per production alternative-machine relationship",
        "primary_sources": [
            "alternative_machine"
        ],
        "analytical_purpose":
            "Connect production alternatives to machines and "
            "their observed production-time requirements."
    },

    {
        "gold_table": "bridge_product_packing",
        "table_type": "BRIDGE",
        "grain": "One row per finished product-packing machine relationship",
        "primary_sources": [
            "product_packing_machine"
        ],
        "analytical_purpose":
            "Connect finished products to packing machines "
            "and observed packing-time requirements."
    },

    {
        "gold_table": "bridge_product_raw_material",
        "table_type": "BRIDGE",
        "grain": "One row per product with source raw-material requirements",
        "primary_sources": [
            "product_raw_material_requirement"
        ],
        "analytical_purpose":
            "Support product-level raw-material requirement analysis."
    }
]


# ----------------------------------------------------------------
# 2. Display proposed Gold model
# ----------------------------------------------------------------

for model in gold_model:

    print(f"Gold table:          {model['gold_table']}")
    print(f"Type:                {model['table_type']}")
    print(f"Grain:               {model['grain']}")
    print(
        f"Silver source(s):    "
        f"{', '.join(model['primary_sources'])}"
    )
    print(
        f"Purpose:             "
        f"{model['analytical_purpose']}"
    )
    print("-" * 120)


# ----------------------------------------------------------------
# 3. Verify required Silver source tables
# ----------------------------------------------------------------

required_silver_tables = sorted(
    {
        source
        for model in gold_model
        for source in model["primary_sources"]
    }
)

available_silver_tables = {
    row["tableName"]
    for row in spark.sql(
        f"SHOW TABLES IN {silver_catalog}.{silver_schema}"
    ).select("tableName").collect()
}

missing_silver_tables = sorted(
    set(required_silver_tables)
    - available_silver_tables
)

source_validation_status = (
    "PASS"
    if len(missing_silver_tables) == 0
    else "FAIL"
)


# ----------------------------------------------------------------
# 4. Model summary
# ----------------------------------------------------------------

dimension_count = sum(
    1
    for model in gold_model
    if model["table_type"] == "DIMENSION"
)

fact_count = sum(
    1
    for model in gold_model
    if model["table_type"] == "FACT"
)

bridge_count = sum(
    1
    for model in gold_model
    if model["table_type"] == "BRIDGE"
)


print()
print("=" * 120)
print("PROPOSED GOLD MODEL SUMMARY")
print("-" * 120)

print(
    f"Gold target schema:          "
    f"{gold_catalog}.{gold_schema}"
)

print(
    f"Proposed Gold tables:        "
    f"{len(gold_model)}"
)

print(
    f"Dimensions:                  "
    f"{dimension_count}"
)

print(
    f"Facts:                       "
    f"{fact_count}"
)

print(
    f"Bridges:                     "
    f"{bridge_count}"
)

print(
    f"Required Silver sources:     "
    f"{len(required_silver_tables)}"
)

print(
    f"Missing Silver sources:      "
    f"{len(missing_silver_tables)}"
)

print(
    f"Source availability:         "
    f"{source_validation_status}"
)

if missing_silver_tables:

    print(
        "Missing tables:              "
        + ", ".join(missing_silver_tables)
    )

print("=" * 120)


if source_validation_status == "PASS":

    print(
        "GOLD MODEL DESIGN STATUS: DEFINED — "
        "all proposed Gold model source tables are available "
        "in the validated Silver layer."
    )

    print(
        "No Gold tables have been created yet."
    )

else:

    raise ValueError(
        "GOLD MODEL DESIGN FAILED: one or more required "
        "Silver source tables are unavailable."
    )

GOLD DATA WAREHOUSE — ANALYTICAL MODEL DESIGN
Gold table:          dim_product
Type:                DIMENSION
Grain:               One row per product
Silver source(s):    finished_product_type, finished_product_economics, product_inventory, product_lot_size
Purpose:             Central product attributes, economics, inventory, and lot-size context.
------------------------------------------------------------------------------------------------------------------------
Gold table:          dim_planning_period
Type:                DIMENSION
Grain:               One row per planning period
Silver source(s):    planning_periods
Purpose:             Planning-horizon period structure for demand and capacity analysis.
------------------------------------------------------------------------------------------------------------------------
Gold table:          dim_machine
Type:                DIMENSION
Grain:               One row per production machine
Silver source(s):    machine
Purpose:     

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Gold Data Warehouse — Step 02:
# Validate Gold Grain and Join Cardinality
#
# Purpose:
# Verify that proposed Gold tables can be constructed at their
# intended analytical grain without unintended row multiplication.
#
# No Gold tables are created or modified in this step.

from pyspark.sql import functions as F

print("GOLD DATA WAREHOUSE — GRAIN & JOIN CARDINALITY VALIDATION")
print("=" * 120)

silver_catalog = "workspace"
silver_schema = "silver_sop"

silver_prefix = (
    f"{silver_catalog}.{silver_schema}"
)


# ----------------------------------------------------------------
# 1. Define grain tests
# ----------------------------------------------------------------

grain_tests = [

    (
        "finished_product_type",
        ["finished_product"],
        "dim_product"
    ),

    (
        "finished_product_economics",
        ["finished_product"],
        "dim_product"
    ),

    (
        "product_inventory",
        ["product"],
        "dim_product"
    ),

    (
        "product_lot_size",
        ["product"],
        "dim_product"
    ),

    (
        "planning_periods",
        ["period"],
        "dim_planning_period"
    ),

    (
        "machine",
        ["machine"],
        "dim_machine"
    ),

    (
        "packing_machine",
        ["packing_machine"],
        "dim_packing_machine"
    ),

    (
        "customer_order",
        ["order_id"],
        "fact_customer_order"
    ),

    (
        "demand_plan",
        ["finished_product", "period"],
        "fact_demand_plan"
    ),

    (
        "machine_capacity",
        ["machine", "period"],
        "fact_machine_capacity"
    ),

    (
        "packing_capacity",
        ["packing_machine", "period"],
        "fact_packing_capacity"
    ),

    (
        "product_production_alternative",
        ["product", "alternative"],
        "bridge_product_production"
    ),

    (
        "alternative_machine",
        ["alternative", "machine"],
        "bridge_alternative_machine"
    ),

    (
        "product_packing_machine",
        ["finished_product", "packing_machine"],
        "bridge_product_packing"
    ),

    (
        "product_raw_material_requirement",
        ["product"],
        "bridge_product_raw_material"
    )
]


# ----------------------------------------------------------------
# 2. Execute grain tests
# ----------------------------------------------------------------

grain_results = []

for (
    source_table,
    key_columns,
    gold_target
) in grain_tests:

    df = spark.table(
        f"{silver_prefix}.{source_table}"
    )

    row_count = df.count()

    distinct_key_count = (
        df
        .select(*key_columns)
        .distinct()
        .count()
    )

    duplicate_key_rows = (
        row_count - distinct_key_count
    )

    status = (
        "PASS"
        if duplicate_key_rows == 0
        else "FAIL"
    )

    grain_results.append(
        {
            "gold_target": gold_target,
            "source_table": source_table,
            "key_columns": ", ".join(key_columns),
            "row_count": row_count,
            "distinct_key_count": distinct_key_count,
            "duplicate_key_rows": duplicate_key_rows,
            "status": status
        }
    )


grain_results_df = spark.createDataFrame(
    grain_results
)

display(
    grain_results_df
    .orderBy(
        "gold_target",
        "source_table"
    )
)


# ----------------------------------------------------------------
# 3. Validate dim_product join cardinality
#
# Product domains are not assumed to be identical.
# We inspect actual coverage before deciding the dimension base.
# ----------------------------------------------------------------

product_type_df = (
    spark.table(
        f"{silver_prefix}.finished_product_type"
    )
    .select(
        F.col("finished_product")
        .alias("product")
    )
    .distinct()
)

economics_df = (
    spark.table(
        f"{silver_prefix}.finished_product_economics"
    )
    .select(
        F.col("finished_product")
        .alias("product")
    )
    .distinct()
)

inventory_df = (
    spark.table(
        f"{silver_prefix}.product_inventory"
    )
    .select("product")
    .distinct()
)

lot_size_df = (
    spark.table(
        f"{silver_prefix}.product_lot_size"
    )
    .select("product")
    .distinct()
)


# ----------------------------------------------------------------
# 4. Build union of observed product identifiers
#
# This is for validation only.
# No Gold dimension is persisted here.
# ----------------------------------------------------------------

all_product_keys_df = (
    product_type_df
    .union(economics_df)
    .union(inventory_df)
    .union(lot_size_df)
    .distinct()
)

all_product_count = (
    all_product_keys_df.count()
)


# ----------------------------------------------------------------
# 5. Measure source coverage against observed product universe
# ----------------------------------------------------------------

product_source_dfs = {
    "finished_product_type": product_type_df,
    "finished_product_economics": economics_df,
    "product_inventory": inventory_df,
    "product_lot_size": lot_size_df
}

coverage_results = []

for source_name, source_df in product_source_dfs.items():

    source_products = source_df.count()

    missing_from_source = (
        all_product_keys_df
        .join(
            source_df,
            on="product",
            how="left_anti"
        )
        .count()
    )

    coverage_results.append(
        {
            "source_table": source_name,
            "distinct_products": source_products,
            "observed_product_universe": all_product_count,
            "products_missing_from_source": missing_from_source
        }
    )


coverage_results_df = spark.createDataFrame(
    coverage_results
)

print()
print("PRODUCT DOMAIN COVERAGE")
print("-" * 120)

display(
    coverage_results_df
    .orderBy("source_table")
)


# ----------------------------------------------------------------
# 6. Overall validation
# ----------------------------------------------------------------

failed_grain_tests = sum(
    1
    for result in grain_results
    if result["status"] == "FAIL"
)

grain_status = (
    "PASS"
    if failed_grain_tests == 0
    else "FAIL"
)


print()
print("=" * 120)
print("GOLD GRAIN VALIDATION SUMMARY")
print("-" * 120)

print(
    f"Grain tests executed:          "
    f"{len(grain_results)}"
)

print(
    f"Grain tests passed:            "
    f"{len(grain_results) - failed_grain_tests}"
)

print(
    f"Grain tests failed:            "
    f"{failed_grain_tests}"
)

print(
    f"Observed product universe:     "
    f"{all_product_count}"
)

print(
    f"Grain validation status:       "
    f"{grain_status}"
)

print("=" * 120)


if grain_status == "PASS":

    print(
        "GOLD GRAIN VALIDATION STATUS: PASS — "
        "all proposed Gold source grains are unique at "
        "their intended keys."
    )

    print(
        "Product-domain coverage has been profiled and "
        "must be reviewed before constructing dim_product."
    )

else:

    raise ValueError(
        "GOLD GRAIN VALIDATION FAILED: one or more "
        "proposed Gold source grains are not unique."
    )

GOLD DATA WAREHOUSE — GRAIN & JOIN CARDINALITY VALIDATION


distinct_key_count,duplicate_key_rows,gold_target,key_columns,row_count,source_table,status
734,0,bridge_alternative_machine,"alternative, machine",734,alternative_machine,PASS
189,0,bridge_product_packing,"finished_product, packing_machine",189,product_packing_machine,PASS
592,0,bridge_product_production,"product, alternative",592,product_production_alternative,PASS
292,0,bridge_product_raw_material,product,292,product_raw_material_requirement,PASS
39,0,dim_machine,machine,39,machine,PASS
7,0,dim_packing_machine,packing_machine,7,packing_machine,PASS
8,0,dim_planning_period,period,8,planning_periods,PASS
67,0,dim_product,finished_product,67,finished_product_economics,PASS
67,0,dim_product,finished_product,67,finished_product_type,PASS
292,0,dim_product,product,292,product_inventory,PASS



PRODUCT DOMAIN COVERAGE
------------------------------------------------------------------------------------------------------------------------


distinct_products,observed_product_universe,products_missing_from_source,source_table
67,292,225,finished_product_economics
67,292,225,finished_product_type
292,292,0,product_inventory
292,292,0,product_lot_size



GOLD GRAIN VALIDATION SUMMARY
------------------------------------------------------------------------------------------------------------------------
Grain tests executed:          15
Grain tests passed:            15
Grain tests failed:            0
Observed product universe:     292
Grain validation status:       PASS
GOLD GRAIN VALIDATION STATUS: PASS — all proposed Gold source grains are unique at their intended keys.
Product-domain coverage has been profiled and must be reviewed before constructing dim_product.


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Gold Data Warehouse — Step 03:
# Build and Validate dim_product
#
# Purpose:
# Construct the Gold product dimension using the complete
# 292-product domain evidenced by the Silver layer.
#
# Design:
# - Base population = product_inventory (292 unique products)
# - product_lot_size = complete 292-product coverage
# - finished_product_type = attributes available for 67 products
# - finished_product_economics = economics available for 67 products
#
# LEFT JOIN is used for finished-product attributes so that
# products without those source attributes are not discarded.
#
# No Gold Delta table is persisted in this step.

from pyspark.sql import functions as F

print("GOLD DATA WAREHOUSE — BUILD & VALIDATE dim_product")
print("=" * 120)

silver_catalog = "workspace"
silver_schema = "silver_sop"

silver_prefix = (
    f"{silver_catalog}.{silver_schema}"
)


# ----------------------------------------------------------------
# 1. Load validated Silver sources
# ----------------------------------------------------------------

inventory_df = (
    spark.table(
        f"{silver_prefix}.product_inventory"
    )
    .select(
        "product",
        "initial_inventory_m"
    )
)

lot_size_df = (
    spark.table(
        f"{silver_prefix}.product_lot_size"
    )
    .select(
        "product",
        "minimum_lot_size_m"
    )
)

product_type_df = (
    spark.table(
        f"{silver_prefix}.finished_product_type"
    )
    .select(
        F.col("finished_product").alias("product"),
        "mts_mto"
    )
)

economics_df = (
    spark.table(
        f"{silver_prefix}.finished_product_economics"
    )
    .select(
        F.col("finished_product").alias("product"),
        "industrial_cost_eur_m",
        "margin_pct",
        "average_sales_price_eur_m"
    )
)


# ----------------------------------------------------------------
# 2. Construct Gold product dimension
# ----------------------------------------------------------------

dim_product_df = (
    inventory_df.alias("i")

    .join(
        lot_size_df.alias("l"),
        on="product",
        how="left"
    )

    .join(
        product_type_df.alias("t"),
        on="product",
        how="left"
    )

    .join(
        economics_df.alias("e"),
        on="product",
        how="left"
    )

    .select(
        "product",
        "mts_mto",
        "initial_inventory_m",
        "minimum_lot_size_m",
        "industrial_cost_eur_m",
        "margin_pct",
        "average_sales_price_eur_m"
    )
)


# ----------------------------------------------------------------
# 3. Validate intended Gold grain
# ----------------------------------------------------------------

row_count = dim_product_df.count()

distinct_product_count = (
    dim_product_df
    .select("product")
    .distinct()
    .count()
)

duplicate_product_rows = (
    row_count - distinct_product_count
)


# ----------------------------------------------------------------
# 4. Validate attribute coverage
# ----------------------------------------------------------------

coverage_summary = (
    dim_product_df
    .agg(

        F.count("*")
        .alias("total_products"),

        F.count("mts_mto")
        .alias("products_with_type"),

        F.count("initial_inventory_m")
        .alias("products_with_inventory"),

        F.count("minimum_lot_size_m")
        .alias("products_with_lot_size"),

        F.count("industrial_cost_eur_m")
        .alias("products_with_economics")

    )
)

coverage = coverage_summary.first()


# ----------------------------------------------------------------
# 5. Validate alignment of the two 67-product sources
# ----------------------------------------------------------------

type_products = (
    product_type_df
    .select("product")
)

economics_products = (
    economics_df
    .select("product")
)

type_without_economics = (
    type_products
    .join(
        economics_products,
        on="product",
        how="left_anti"
    )
    .count()
)

economics_without_type = (
    economics_products
    .join(
        type_products,
        on="product",
        how="left_anti"
    )
    .count()
)


# ----------------------------------------------------------------
# 6. Validate full-coverage source joins
# ----------------------------------------------------------------

missing_lot_size = (
    dim_product_df
    .filter(
        F.col("minimum_lot_size_m").isNull()
    )
    .count()
)

missing_inventory = (
    dim_product_df
    .filter(
        F.col("initial_inventory_m").isNull()
    )
    .count()
)


# ----------------------------------------------------------------
# 7. Determine validation status
# ----------------------------------------------------------------

grain_status = (
    "PASS"
    if (
        row_count == 292
        and distinct_product_count == 292
        and duplicate_product_rows == 0
    )
    else "FAIL"
)

full_coverage_status = (
    "PASS"
    if (
        missing_inventory == 0
        and missing_lot_size == 0
    )
    else "FAIL"
)

finished_product_alignment_status = (
    "PASS"
    if (
        type_without_economics == 0
        and economics_without_type == 0
    )
    else "REVIEW"
)


# ----------------------------------------------------------------
# 8. Display analytical preview
# ----------------------------------------------------------------

print("dim_product PREVIEW")
print("-" * 120)

display(
    dim_product_df
    .orderBy("product")
    .limit(20)
)


# ----------------------------------------------------------------
# 9. Validation summary
# ----------------------------------------------------------------

print()
print("=" * 120)
print("dim_product VALIDATION SUMMARY")
print("-" * 120)

print(
    f"Gold dimension rows:              "
    f"{row_count}"
)

print(
    f"Distinct products:                "
    f"{distinct_product_count}"
)

print(
    f"Duplicate product rows:           "
    f"{duplicate_product_rows}"
)

print(
    f"Products with inventory:          "
    f"{coverage['products_with_inventory']}"
)

print(
    f"Products with lot size:           "
    f"{coverage['products_with_lot_size']}"
)

print(
    f"Products with type (MTS/MTO):     "
    f"{coverage['products_with_type']}"
)

print(
    f"Products with economics:          "
    f"{coverage['products_with_economics']}"
)

print(
    f"Type without economics:           "
    f"{type_without_economics}"
)

print(
    f"Economics without type:           "
    f"{economics_without_type}"
)

print(
    f"Gold grain validation:            "
    f"{grain_status}"
)

print(
    f"Full-domain attribute coverage:   "
    f"{full_coverage_status}"
)

print(
    f"Finished-product alignment:       "
    f"{finished_product_alignment_status}"
)

print("=" * 120)


# ----------------------------------------------------------------
# 10. Final control
# ----------------------------------------------------------------

if (
    grain_status == "PASS"
    and full_coverage_status == "PASS"
):

    print(
        "dim_product BUILD STATUS: VALID — "
        "the 292-product Gold dimension preserves the complete "
        "observed product domain without row multiplication."
    )

    if finished_product_alignment_status == "PASS":

        print(
            "Finished-product type and economics domains are "
            "fully aligned."
        )

    else:

        print(
            "REVIEW: finished-product type and economics domains "
            "are not identical. No source values were fabricated."
        )

    print(
        "dim_product has NOT yet been persisted as a Gold Delta table."
    )

else:

    raise ValueError(
        "dim_product VALIDATION FAILED: Gold grain or complete-domain "
        "coverage does not reconcile with the validated Silver sources."
    )

GOLD DATA WAREHOUSE — BUILD & VALIDATE dim_product
dim_product PREVIEW
------------------------------------------------------------------------------------------------------------------------


product,mts_mto,initial_inventory_m,minimum_lot_size_m,industrial_cost_eur_m,margin_pct,average_sales_price_eur_m
6182506,MTS,70,3000,0.86,0.28,1.1008
6182511,MTS,8151,2000,1.36,0.28,1.7408000000000001
6182512,MTS,1109,4000,2.07,0.31,2.7117
6182513,MTS,1927,3000,1.7,0.21,2.057
6182514,MTS,286,6000,1.34,0.29,1.7286000000000001
6182515,MTS,2665,6000,1.09,0.28,1.3952000000000002
6182516,MTS,32112,10000,0.9,0.24,1.116
6182521,MTS,12597,3000,2.63,0.28,3.3664
6182733,MTS,3571,6000,0.95,0.28,1.216
6182736,MTS,5368,2000,1.17,0.26,1.4742



dim_product VALIDATION SUMMARY
------------------------------------------------------------------------------------------------------------------------
Gold dimension rows:              292
Distinct products:                292
Duplicate product rows:           0
Products with inventory:          292
Products with lot size:           292
Products with type (MTS/MTO):     67
Products with economics:          67
Type without economics:           0
Economics without type:           0
Gold grain validation:            PASS
Full-domain attribute coverage:   PASS
Finished-product alignment:       PASS
dim_product BUILD STATUS: VALID — the 292-product Gold dimension preserves the complete observed product domain without row multiplication.
Finished-product type and economics domains are fully aligned.
dim_product has NOT yet been persisted as a Gold Delta table.


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Gold Data Warehouse — Step 04:
# Build and Validate Reference Dimensions
#
# Purpose:
# Construct the remaining Gold reference dimensions from
# validated Silver sources and verify their intended grain.
#
# Dimensions:
# - dim_planning_period
# - dim_machine
# - dim_packing_machine
#
# No Gold Delta tables are persisted in this step.

from pyspark.sql import functions as F

print("GOLD DATA WAREHOUSE — BUILD & VALIDATE REFERENCE DIMENSIONS")
print("=" * 120)

silver_catalog = "workspace"
silver_schema = "silver_sop"

silver_prefix = f"{silver_catalog}.{silver_schema}"


# ----------------------------------------------------------------
# 1. Build dim_planning_period
# ----------------------------------------------------------------

dim_planning_period_df = (
    spark.table(
        f"{silver_prefix}.planning_periods"
    )
    .select(
        "period",
        "macroperiod_month",
        "last_day_of_period",
        "number_of_days"
    )
)


# ----------------------------------------------------------------
# 2. Build dim_machine
# ----------------------------------------------------------------

dim_machine_df = (
    spark.table(
        f"{silver_prefix}.machine"
    )
    .select(
        "machine",
        "cost_per_unit_of_time_eur_sec"
    )
)


# ----------------------------------------------------------------
# 3. Build dim_packing_machine
# ----------------------------------------------------------------

dim_packing_machine_df = (
    spark.table(
        f"{silver_prefix}.packing_machine"
    )
    .select(
        "packing_machine",
        "cost_per_unit_of_time_eur_sec"
    )
)


# ----------------------------------------------------------------
# 4. Define dimension validation tests
# ----------------------------------------------------------------

dimension_tests = [
    {
        "dimension": "dim_planning_period",
        "df": dim_planning_period_df,
        "key": ["period"],
        "expected_rows": 8
    },
    {
        "dimension": "dim_machine",
        "df": dim_machine_df,
        "key": ["machine"],
        "expected_rows": 39
    },
    {
        "dimension": "dim_packing_machine",
        "df": dim_packing_machine_df,
        "key": ["packing_machine"],
        "expected_rows": 7
    }
]


# ----------------------------------------------------------------
# 5. Execute dimension grain validation
# ----------------------------------------------------------------

dimension_results = []

for test in dimension_tests:

    df = test["df"]
    key_columns = test["key"]
    expected_rows = test["expected_rows"]

    row_count = df.count()

    distinct_key_count = (
        df
        .select(*key_columns)
        .distinct()
        .count()
    )

    duplicate_key_rows = (
        row_count - distinct_key_count
    )

    null_key_rows = (
        df
        .filter(
            F.expr(
                " OR ".join(
                    [
                        f"`{column}` IS NULL"
                        for column in key_columns
                    ]
                )
            )
        )
        .count()
    )

    status = (
        "PASS"
        if (
            row_count == expected_rows
            and distinct_key_count == expected_rows
            and duplicate_key_rows == 0
            and null_key_rows == 0
        )
        else "FAIL"
    )

    dimension_results.append(
        {
            "dimension": test["dimension"],
            "expected_rows": expected_rows,
            "actual_rows": row_count,
            "distinct_keys": distinct_key_count,
            "duplicate_key_rows": duplicate_key_rows,
            "null_key_rows": null_key_rows,
            "status": status
        }
    )


# ----------------------------------------------------------------
# 6. Display validation results
# ----------------------------------------------------------------

dimension_results_df = spark.createDataFrame(
    dimension_results
)

display(
    dimension_results_df
    .orderBy("dimension")
)


# ----------------------------------------------------------------
# 7. Validate planning-period structure
# ----------------------------------------------------------------

period_rows = dim_planning_period_df.count()

period_min = (
    dim_planning_period_df
    .agg(F.min("period"))
    .first()[0]
)

period_max = (
    dim_planning_period_df
    .agg(F.max("period"))
    .first()[0]
)

total_period_days = (
    dim_planning_period_df
    .agg(F.sum("number_of_days"))
    .first()[0]
)

horizon_end_day = (
    dim_planning_period_df
    .agg(F.max("last_day_of_period"))
    .first()[0]
)

planning_structure_status = (
    "PASS"
    if (
        period_rows == 8
        and period_min == 1
        and period_max == 8
        and total_period_days == 101
        and horizon_end_day == 101
    )
    else "FAIL"
)


# ----------------------------------------------------------------
# 8. Overall validation
# ----------------------------------------------------------------

failed_dimensions = sum(
    1
    for result in dimension_results
    if result["status"] == "FAIL"
)

dimension_status = (
    "PASS"
    if failed_dimensions == 0
    else "FAIL"
)


print()
print("=" * 120)
print("REFERENCE DIMENSION VALIDATION SUMMARY")
print("-" * 120)

print(
    f"Dimensions constructed:           "
    f"{len(dimension_results)}"
)

print(
    f"Dimensions passed:                "
    f"{len(dimension_results) - failed_dimensions}"
)

print(
    f"Dimensions failed:                "
    f"{failed_dimensions}"
)

print()
print("PLANNING PERIOD STRUCTURE")
print("-" * 120)

print(
    f"Planning periods:                 "
    f"{period_rows}"
)

print(
    f"Period range:                     "
    f"{period_min} through {period_max}"
)

print(
    f"Total planning-horizon days:      "
    f"{total_period_days}"
)

print(
    f"Final planning-horizon day:       "
    f"{horizon_end_day}"
)

print(
    f"Planning structure validation:    "
    f"{planning_structure_status}"
)

print()
print(
    f"Reference dimension validation:   "
    f"{dimension_status}"
)

print("=" * 120)


# ----------------------------------------------------------------
# 9. Final control
# ----------------------------------------------------------------

if (
    dimension_status == "PASS"
    and planning_structure_status == "PASS"
):

    print(
        "REFERENCE DIMENSION BUILD STATUS: VALID — "
        "all three Gold reference dimensions preserve "
        "their validated Silver grain and planning structure."
    )

    print(
        "Gold reference dimensions have NOT yet been "
        "persisted as Delta tables."
    )

else:

    raise ValueError(
        "REFERENCE DIMENSION VALIDATION FAILED: "
        "one or more Gold dimensions do not reconcile "
        "with the validated Silver sources."
    )

GOLD DATA WAREHOUSE — BUILD & VALIDATE REFERENCE DIMENSIONS


actual_rows,dimension,distinct_keys,duplicate_key_rows,expected_rows,null_key_rows,status
39,dim_machine,39,0,39,0,PASS
7,dim_packing_machine,7,0,7,0,PASS
8,dim_planning_period,8,0,8,0,PASS



REFERENCE DIMENSION VALIDATION SUMMARY
------------------------------------------------------------------------------------------------------------------------
Dimensions constructed:           3
Dimensions passed:                3
Dimensions failed:                0

PLANNING PERIOD STRUCTURE
------------------------------------------------------------------------------------------------------------------------
Planning periods:                 8
Period range:                     1 through 8
Total planning-horizon days:      101
Final planning-horizon day:       101
Planning structure validation:    PASS

Reference dimension validation:   PASS
REFERENCE DIMENSION BUILD STATUS: VALID — all three Gold reference dimensions preserve their validated Silver grain and planning structure.
Gold reference dimensions have NOT yet been persisted as Delta tables.


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Gold Data Warehouse — Step 05:
# Build and Validate Gold Fact Tables
#
# Purpose:
# Construct the four proposed Gold fact tables from validated
# Silver sources and verify:
# - intended analytical grain
# - row-count preservation
# - foreign-key coverage against Gold dimensions
#
# No Gold Delta tables are persisted in this step.

from pyspark.sql import functions as F

print("GOLD DATA WAREHOUSE — BUILD & VALIDATE FACT TABLES")
print("=" * 120)

silver_catalog = "workspace"
silver_schema = "silver_sop"

silver_prefix = f"{silver_catalog}.{silver_schema}"


# ----------------------------------------------------------------
# 1. Build fact_customer_order
# ----------------------------------------------------------------

fact_customer_order_df = (
    spark.table(
        f"{silver_prefix}.customer_order"
    )
    .select(
        "order_id",
        F.col("finished_product").alias("product"),
        "due_date_days_in_the_planning_horizon",
        "quantity_m",
        "sales_value_eur",
        "planning_horizon_status"
    )
)


# ----------------------------------------------------------------
# 2. Build fact_demand_plan
# ----------------------------------------------------------------

fact_demand_plan_df = (
    spark.table(
        f"{silver_prefix}.demand_plan"
    )
    .select(
        F.col("finished_product").alias("product"),
        "period",
        "adjusted_total_demand_m",
        "target_stock_m"
    )
)


# ----------------------------------------------------------------
# 3. Build fact_machine_capacity
# ----------------------------------------------------------------

fact_machine_capacity_df = (
    spark.table(
        f"{silver_prefix}.machine_capacity"
    )
    .select(
        "machine",
        "period",
        "available_regular_time_h",
        "available_overtime_h"
    )
)


# ----------------------------------------------------------------
# 4. Build fact_packing_capacity
# ----------------------------------------------------------------

fact_packing_capacity_df = (
    spark.table(
        f"{silver_prefix}.packing_capacity"
    )
    .select(
        "packing_machine",
        "period",
        "available_regular_time_h",
        "available_overtime_h"
    )
)


# ----------------------------------------------------------------
# 5. Define fact grain validation
# ----------------------------------------------------------------

fact_tests = [

    {
        "fact": "fact_customer_order",
        "df": fact_customer_order_df,
        "key": ["order_id"],
        "expected_rows": 61
    },

    {
        "fact": "fact_demand_plan",
        "df": fact_demand_plan_df,
        "key": ["product", "period"],
        "expected_rows": 246
    },

    {
        "fact": "fact_machine_capacity",
        "df": fact_machine_capacity_df,
        "key": ["machine", "period"],
        "expected_rows": 312
    },

    {
        "fact": "fact_packing_capacity",
        "df": fact_packing_capacity_df,
        "key": ["packing_machine", "period"],
        "expected_rows": 56
    }
]


fact_results = []

for test in fact_tests:

    df = test["df"]

    row_count = df.count()

    distinct_key_count = (
        df
        .select(*test["key"])
        .distinct()
        .count()
    )

    duplicate_key_rows = (
        row_count - distinct_key_count
    )

    status = (
        "PASS"
        if (
            row_count == test["expected_rows"]
            and distinct_key_count == row_count
            and duplicate_key_rows == 0
        )
        else "FAIL"
    )

    fact_results.append(
        {
            "fact": test["fact"],
            "expected_rows": test["expected_rows"],
            "actual_rows": row_count,
            "distinct_keys": distinct_key_count,
            "duplicate_key_rows": duplicate_key_rows,
            "status": status
        }
    )


fact_results_df = spark.createDataFrame(
    fact_results
)

display(
    fact_results_df
    .orderBy("fact")
)


# ----------------------------------------------------------------
# 6. Foreign-key validation helper
# ----------------------------------------------------------------

def orphan_count(
    child_df,
    child_column,
    parent_df,
    parent_column
):

    child_keys = (
        child_df
        .select(
            F.col(child_column).alias("join_key")
        )
        .filter(
            F.col("join_key").isNotNull()
        )
        .distinct()
    )

    parent_keys = (
        parent_df
        .select(
            F.col(parent_column).alias("join_key")
        )
        .filter(
            F.col("join_key").isNotNull()
        )
        .distinct()
    )

    return (
        child_keys
        .join(
            parent_keys,
            on="join_key",
            how="left_anti"
        )
        .count()
    )


# ----------------------------------------------------------------
# 7. Validate fact → dimension relationships
# ----------------------------------------------------------------

relationship_results = []


relationships = [

    (
        "fact_customer_order",
        fact_customer_order_df,
        "product",
        "dim_product",
        dim_product_df,
        "product"
    ),

    (
        "fact_demand_plan",
        fact_demand_plan_df,
        "product",
        "dim_product",
        dim_product_df,
        "product"
    ),

    (
        "fact_demand_plan",
        fact_demand_plan_df,
        "period",
        "dim_planning_period",
        dim_planning_period_df,
        "period"
    ),

    (
        "fact_machine_capacity",
        fact_machine_capacity_df,
        "machine",
        "dim_machine",
        dim_machine_df,
        "machine"
    ),

    (
        "fact_machine_capacity",
        fact_machine_capacity_df,
        "period",
        "dim_planning_period",
        dim_planning_period_df,
        "period"
    ),

    (
        "fact_packing_capacity",
        fact_packing_capacity_df,
        "packing_machine",
        "dim_packing_machine",
        dim_packing_machine_df,
        "packing_machine"
    ),

    (
        "fact_packing_capacity",
        fact_packing_capacity_df,
        "period",
        "dim_planning_period",
        dim_planning_period_df,
        "period"
    )
]


for (
    fact_name,
    fact_df,
    fact_column,
    dimension_name,
    dimension_df,
    dimension_column
) in relationships:

    orphan_keys = orphan_count(
        fact_df,
        fact_column,
        dimension_df,
        dimension_column
    )

    relationship_results.append(
        {
            "fact": fact_name,
            "fact_key": fact_column,
            "dimension": dimension_name,
            "dimension_key": dimension_column,
            "orphan_keys": orphan_keys,
            "status": (
                "PASS"
                if orphan_keys == 0
                else "FAIL"
            )
        }
    )


relationship_results_df = spark.createDataFrame(
    relationship_results
)

print()
print("FACT → DIMENSION RELATIONSHIP VALIDATION")
print("-" * 120)

display(
    relationship_results_df
    .orderBy(
        "fact",
        "dimension"
    )
)


# ----------------------------------------------------------------
# 8. Preserve customer-order horizon classification
# ----------------------------------------------------------------

horizon_distribution_df = (
    fact_customer_order_df
    .groupBy("planning_horizon_status")
    .count()
    .orderBy("planning_horizon_status")
)

print()
print("CUSTOMER ORDER HORIZON DISTRIBUTION")
print("-" * 120)

display(horizon_distribution_df)


before_horizon_count = (
    fact_customer_order_df
    .filter(
        F.col("planning_horizon_status")
        == "Before horizon start"
    )
    .count()
)

within_horizon_count = (
    fact_customer_order_df
    .filter(
        F.col("planning_horizon_status")
        == "Within horizon"
    )
    .count()
)

beyond_horizon_count = (
    fact_customer_order_df
    .filter(
        F.col("planning_horizon_status")
        == "Beyond horizon end"
    )
    .count()
)


# ----------------------------------------------------------------
# 9. Overall validation
# ----------------------------------------------------------------

failed_fact_tests = sum(
    1
    for result in fact_results
    if result["status"] == "FAIL"
)

failed_relationships = sum(
    1
    for result in relationship_results
    if result["status"] == "FAIL"
)

fact_grain_status = (
    "PASS"
    if failed_fact_tests == 0
    else "FAIL"
)

relationship_status = (
    "PASS"
    if failed_relationships == 0
    else "FAIL"
)

horizon_status = (
    "PASS"
    if (
        before_horizon_count == 9
        and within_horizon_count == 52
        and beyond_horizon_count == 0
    )
    else "FAIL"
)


print()
print("=" * 120)
print("GOLD FACT TABLE VALIDATION SUMMARY")
print("-" * 120)

print(
    f"Fact tables constructed:          "
    f"{len(fact_results)}"
)

print(
    f"Fact grain tests passed:          "
    f"{len(fact_results) - failed_fact_tests}"
    f"/{len(fact_results)}"
)

print(
    f"Fact grain tests failed:          "
    f"{failed_fact_tests}"
)

print(
    f"Fact-dimension relationships:     "
    f"{len(relationship_results)}"
)

print(
    f"Relationships passed:             "
    f"{len(relationship_results) - failed_relationships}"
    f"/{len(relationship_results)}"
)

print(
    f"Relationship validation:          "
    f"{relationship_status}"
)

print()
print("CUSTOMER ORDER HORIZON CONTROL")
print("-" * 120)

print(
    f"Before horizon start:             "
    f"{before_horizon_count}"
)

print(
    f"Within horizon:                   "
    f"{within_horizon_count}"
)

print(
    f"Beyond horizon end:               "
    f"{beyond_horizon_count}"
)

print(
    f"Horizon classification control:   "
    f"{horizon_status}"
)

print()
print(
    f"Overall fact grain validation:    "
    f"{fact_grain_status}"
)

print("=" * 120)


# ----------------------------------------------------------------
# 10. Final control
# ----------------------------------------------------------------

if (
    fact_grain_status == "PASS"
    and relationship_status == "PASS"
    and horizon_status == "PASS"
):

    print(
        "GOLD FACT BUILD STATUS: VALID — "
        "all four Gold fact tables preserve their intended "
        "grain and validated dimensional relationships."
    )

    print(
        "Gold fact tables have NOT yet been persisted "
        "as Delta tables."
    )

else:

    raise ValueError(
        "GOLD FACT VALIDATION FAILED: one or more grain, "
        "relationship, or horizon controls failed."
    )

GOLD DATA WAREHOUSE — BUILD & VALIDATE FACT TABLES


actual_rows,distinct_keys,duplicate_key_rows,expected_rows,fact,status
61,61,0,61,fact_customer_order,PASS
246,246,0,246,fact_demand_plan,PASS
312,312,0,312,fact_machine_capacity,PASS
56,56,0,56,fact_packing_capacity,PASS



FACT → DIMENSION RELATIONSHIP VALIDATION
------------------------------------------------------------------------------------------------------------------------


dimension,dimension_key,fact,fact_key,orphan_keys,status
dim_product,product,fact_customer_order,product,0,PASS
dim_planning_period,period,fact_demand_plan,period,0,PASS
dim_product,product,fact_demand_plan,product,0,PASS
dim_machine,machine,fact_machine_capacity,machine,0,PASS
dim_planning_period,period,fact_machine_capacity,period,0,PASS
dim_packing_machine,packing_machine,fact_packing_capacity,packing_machine,0,PASS
dim_planning_period,period,fact_packing_capacity,period,0,PASS



CUSTOMER ORDER HORIZON DISTRIBUTION
------------------------------------------------------------------------------------------------------------------------


planning_horizon_status,count
Before horizon start,9
Within horizon,52



GOLD FACT TABLE VALIDATION SUMMARY
------------------------------------------------------------------------------------------------------------------------
Fact tables constructed:          4
Fact grain tests passed:          4/4
Fact grain tests failed:          0
Fact-dimension relationships:     7
Relationships passed:             7/7
Relationship validation:          PASS

CUSTOMER ORDER HORIZON CONTROL
------------------------------------------------------------------------------------------------------------------------
Before horizon start:             9
Within horizon:                   52
Beyond horizon end:               0
Horizon classification control:   PASS

Overall fact grain validation:    PASS
GOLD FACT BUILD STATUS: VALID — all four Gold fact tables preserve their intended grain and validated dimensional relationships.
Gold fact tables have NOT yet been persisted as Delta tables.


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Gold Data Warehouse — Step 06:
# Build and Validate Gold Bridge Tables
#
# Purpose:
# Construct the four proposed Gold bridge tables and validate:
# - intended relationship grain
# - source row-count preservation
# - foreign-key coverage against Gold dimensions where applicable
# - preservation of source-coded attributes
#
# No Gold Delta tables are persisted in this step.

from pyspark.sql import functions as F

print("GOLD DATA WAREHOUSE — BUILD & VALIDATE BRIDGE TABLES")
print("=" * 120)

silver_catalog = "workspace"
silver_schema = "silver_sop"

silver_prefix = f"{silver_catalog}.{silver_schema}"


# ----------------------------------------------------------------
# 1. Build bridge_product_production
# ----------------------------------------------------------------

bridge_product_production_df = (
    spark.table(
        f"{silver_prefix}.product_production_alternative"
    )
    .select(
        "product",
        "alternative",
        "main_alternative"
    )
)


# ----------------------------------------------------------------
# 2. Build bridge_alternative_machine
# ----------------------------------------------------------------

bridge_alternative_machine_df = (
    spark.table(
        f"{silver_prefix}.alternative_machine"
    )
    .select(
        "alternative",
        "machine",
        "production_time_sec_m"
    )
)


# ----------------------------------------------------------------
# 3. Build bridge_product_packing
# ----------------------------------------------------------------

bridge_product_packing_df = (
    spark.table(
        f"{silver_prefix}.product_packing_machine"
    )
    .select(
        "finished_product",
        "packing_machine",
        "packing_time_sec_m"
    )
    .withColumnRenamed(
        "finished_product",
        "product"
    )
)


# ----------------------------------------------------------------
# 4. Build bridge_product_raw_material
# ----------------------------------------------------------------

bridge_product_raw_material_df = (
    spark.table(
        f"{silver_prefix}.product_raw_material_requirement"
    )
    .select(
        "product",
        "raw_material_cu_kg_m",
        "raw_material_al_kg_m",
        "raw_material_almelec_kg_m",
        "raw_material_outros_kg_m"
    )
)


# ----------------------------------------------------------------
# 5. Define bridge grain validation
# ----------------------------------------------------------------

bridge_tests = [

    {
        "bridge": "bridge_product_production",
        "df": bridge_product_production_df,
        "key": ["product", "alternative"],
        "expected_rows": 592
    },

    {
        "bridge": "bridge_alternative_machine",
        "df": bridge_alternative_machine_df,
        "key": ["alternative", "machine"],
        "expected_rows": 734
    },

    {
        "bridge": "bridge_product_packing",
        "df": bridge_product_packing_df,
        "key": ["product", "packing_machine"],
        "expected_rows": 189
    },

    {
        "bridge": "bridge_product_raw_material",
        "df": bridge_product_raw_material_df,
        "key": ["product"],
        "expected_rows": 292
    }
]


bridge_results = []

for test in bridge_tests:

    df = test["df"]

    row_count = df.count()

    distinct_key_count = (
        df
        .select(*test["key"])
        .distinct()
        .count()
    )

    duplicate_key_rows = (
        row_count - distinct_key_count
    )

    status = (
        "PASS"
        if (
            row_count == test["expected_rows"]
            and distinct_key_count == row_count
            and duplicate_key_rows == 0
        )
        else "FAIL"
    )

    bridge_results.append(
        {
            "bridge": test["bridge"],
            "expected_rows": test["expected_rows"],
            "actual_rows": row_count,
            "distinct_keys": distinct_key_count,
            "duplicate_key_rows": duplicate_key_rows,
            "status": status
        }
    )


bridge_results_df = spark.createDataFrame(
    bridge_results
)

display(
    bridge_results_df
    .orderBy("bridge")
)


# ----------------------------------------------------------------
# 6. Foreign-key helper
# ----------------------------------------------------------------

def bridge_orphan_count(
    child_df,
    child_column,
    parent_df,
    parent_column
):

    child_keys = (
        child_df
        .select(
            F.col(child_column).alias("join_key")
        )
        .filter(
            F.col("join_key").isNotNull()
        )
        .distinct()
    )

    parent_keys = (
        parent_df
        .select(
            F.col(parent_column).alias("join_key")
        )
        .filter(
            F.col("join_key").isNotNull()
        )
        .distinct()
    )

    return (
        child_keys
        .join(
            parent_keys,
            on="join_key",
            how="left_anti"
        )
        .count()
    )


# ----------------------------------------------------------------
# 7. Validate bridge → dimension relationships
# ----------------------------------------------------------------

bridge_relationships = [

    (
        "bridge_product_production",
        bridge_product_production_df,
        "product",
        "dim_product",
        dim_product_df,
        "product"
    ),

    (
        "bridge_alternative_machine",
        bridge_alternative_machine_df,
        "machine",
        "dim_machine",
        dim_machine_df,
        "machine"
    ),

    (
        "bridge_product_packing",
        bridge_product_packing_df,
        "product",
        "dim_product",
        dim_product_df,
        "product"
    ),

    (
        "bridge_product_packing",
        bridge_product_packing_df,
        "packing_machine",
        "dim_packing_machine",
        dim_packing_machine_df,
        "packing_machine"
    ),

    (
        "bridge_product_raw_material",
        bridge_product_raw_material_df,
        "product",
        "dim_product",
        dim_product_df,
        "product"
    )
]


bridge_relationship_results = []

for (
    bridge_name,
    bridge_df,
    bridge_column,
    dimension_name,
    dimension_df,
    dimension_column
) in bridge_relationships:

    orphan_keys = bridge_orphan_count(
        bridge_df,
        bridge_column,
        dimension_df,
        dimension_column
    )

    bridge_relationship_results.append(
        {
            "bridge": bridge_name,
            "bridge_key": bridge_column,
            "dimension": dimension_name,
            "dimension_key": dimension_column,
            "orphan_keys": orphan_keys,
            "status": (
                "PASS"
                if orphan_keys == 0
                else "FAIL"
            )
        }
    )


bridge_relationship_results_df = spark.createDataFrame(
    bridge_relationship_results
)

print()
print("BRIDGE → DIMENSION RELATIONSHIP VALIDATION")
print("-" * 120)

display(
    bridge_relationship_results_df
    .orderBy(
        "bridge",
        "dimension"
    )
)


# ----------------------------------------------------------------
# 8. Validate alternative relationships
#
# alternative is not promoted to a dimension because the source
# provides relationship-level alternative data rather than a
# separate alternative master table.
# ----------------------------------------------------------------

production_alternatives_df = (
    bridge_product_production_df
    .select("alternative")
    .distinct()
)

machine_alternatives_df = (
    bridge_alternative_machine_df
    .select("alternative")
    .distinct()
)

production_alternative_count = (
    production_alternatives_df.count()
)

machine_alternative_count = (
    machine_alternatives_df.count()
)

production_without_machine = (
    production_alternatives_df
    .join(
        machine_alternatives_df,
        on="alternative",
        how="left_anti"
    )
    .count()
)

machine_without_product = (
    machine_alternatives_df
    .join(
        production_alternatives_df,
        on="alternative",
        how="left_anti"
    )
    .count()
)

alternative_relationship_status = (
    "PASS"
    if (
        production_without_machine == 0
        and machine_without_product == 0
    )
    else "REVIEW"
)


# ----------------------------------------------------------------
# 9. Preserve main_alternative source domain
# ----------------------------------------------------------------

main_alternative_values = [
    row["main_alternative"]
    for row in (
        bridge_product_production_df
        .select("main_alternative")
        .distinct()
        .orderBy("main_alternative")
        .collect()
    )
]

main_alternative_status = (
    "PASS"
    if main_alternative_values == [1, 2]
    else "FAIL"
)


# ----------------------------------------------------------------
# 10. Overall validation
# ----------------------------------------------------------------

failed_bridge_tests = sum(
    1
    for result in bridge_results
    if result["status"] == "FAIL"
)

failed_bridge_relationships = sum(
    1
    for result in bridge_relationship_results
    if result["status"] == "FAIL"
)

bridge_grain_status = (
    "PASS"
    if failed_bridge_tests == 0
    else "FAIL"
)

bridge_relationship_status = (
    "PASS"
    if failed_bridge_relationships == 0
    else "FAIL"
)


print()
print("=" * 120)
print("GOLD BRIDGE TABLE VALIDATION SUMMARY")
print("-" * 120)

print(
    f"Bridge tables constructed:          "
    f"{len(bridge_results)}"
)

print(
    f"Bridge grain tests passed:          "
    f"{len(bridge_results) - failed_bridge_tests}"
    f"/{len(bridge_results)}"
)

print(
    f"Bridge grain tests failed:          "
    f"{failed_bridge_tests}"
)

print(
    f"Bridge-dimension relationships:     "
    f"{len(bridge_relationship_results)}"
)

print(
    f"Relationships passed:               "
    f"{len(bridge_relationship_results) - failed_bridge_relationships}"
    f"/{len(bridge_relationship_results)}"
)

print(
    f"Bridge relationship validation:     "
    f"{bridge_relationship_status}"
)

print()
print("PRODUCTION ALTERNATIVE CONTROL")
print("-" * 120)

print(
    f"Alternatives linked to products:     "
    f"{production_alternative_count}"
)

print(
    f"Alternatives linked to machines:     "
    f"{machine_alternative_count}"
)

print(
    f"Product alternatives without machine:"
    f" {production_without_machine}"
)

print(
    f"Machine alternatives without product:"
    f" {machine_without_product}"
)

print(
    f"Alternative relationship status:     "
    f"{alternative_relationship_status}"
)

print()
print("SOURCE-CODE PRESERVATION")
print("-" * 120)

print(
    f"main_alternative observed domain:     "
    f"{main_alternative_values}"
)

print(
    f"main_alternative preservation:        "
    f"{main_alternative_status}"
)

print()
print(
    f"Overall bridge grain validation:      "
    f"{bridge_grain_status}"
)

print("=" * 120)


# ----------------------------------------------------------------
# 11. Final control
# ----------------------------------------------------------------

if (
    bridge_grain_status == "PASS"
    and bridge_relationship_status == "PASS"
    and main_alternative_status == "PASS"
):

    print(
        "GOLD BRIDGE BUILD STATUS: VALID — "
        "all four Gold bridge tables preserve their intended "
        "relationship grain and validated dimensional references."
    )

    if alternative_relationship_status == "PASS":

        print(
            "Production-alternative coverage is aligned between "
            "product and machine relationships."
        )

    else:

        print(
            "REVIEW: production-alternative coverage differs "
            "between relationship sources. No relationships "
            "were fabricated."
        )

    print(
        "Gold bridge tables have NOT yet been persisted "
        "as Delta tables."
    )

else:

    raise ValueError(
        "GOLD BRIDGE VALIDATION FAILED: one or more grain, "
        "relationship, or source-domain controls failed."
    )

GOLD DATA WAREHOUSE — BUILD & VALIDATE BRIDGE TABLES


actual_rows,bridge,distinct_keys,duplicate_key_rows,expected_rows,status
734,bridge_alternative_machine,734,0,734,PASS
189,bridge_product_packing,189,0,189,PASS
592,bridge_product_production,592,0,592,PASS
292,bridge_product_raw_material,292,0,292,PASS



BRIDGE → DIMENSION RELATIONSHIP VALIDATION
------------------------------------------------------------------------------------------------------------------------


bridge,bridge_key,dimension,dimension_key,orphan_keys,status
bridge_alternative_machine,machine,dim_machine,machine,0,PASS
bridge_product_packing,packing_machine,dim_packing_machine,packing_machine,0,PASS
bridge_product_packing,product,dim_product,product,0,PASS
bridge_product_production,product,dim_product,product,0,PASS
bridge_product_raw_material,product,dim_product,product,0,PASS



GOLD BRIDGE TABLE VALIDATION SUMMARY
------------------------------------------------------------------------------------------------------------------------
Bridge tables constructed:          4
Bridge grain tests passed:          4/4
Bridge grain tests failed:          0
Bridge-dimension relationships:     5
Relationships passed:               5/5
Bridge relationship validation:     PASS

PRODUCTION ALTERNATIVE CONTROL
------------------------------------------------------------------------------------------------------------------------
Alternatives linked to products:     592
Alternatives linked to machines:     583
Product alternatives without machine: 9
Machine alternatives without product: 0
Alternative relationship status:     REVIEW

SOURCE-CODE PRESERVATION
------------------------------------------------------------------------------------------------------------------------
main_alternative observed domain:     [1, 2]
main_alternative preservation:        PASS

Overall bri

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Gold Data Warehouse — Step 07:
# Investigate Unmapped Production Alternatives
#
# Purpose:
# Investigate the nine product-linked production alternatives
# that do not have a corresponding machine relationship.
#
# This is an analytical investigation only.
# No source relationships are deleted, modified, or fabricated.
# No Gold Delta tables are persisted in this step.

from pyspark.sql import functions as F

print("GOLD DATA WAREHOUSE — UNMAPPED PRODUCTION ALTERNATIVE INVESTIGATION")
print("=" * 120)


# ----------------------------------------------------------------
# 1. Identify alternatives used by products
# ----------------------------------------------------------------

product_alternatives_df = (
    bridge_product_production_df
    .select("alternative")
    .distinct()
)


# ----------------------------------------------------------------
# 2. Identify alternatives mapped to machines
# ----------------------------------------------------------------

machine_alternatives_df = (
    bridge_alternative_machine_df
    .select("alternative")
    .distinct()
)


# ----------------------------------------------------------------
# 3. Identify product alternatives with no machine relationship
# ----------------------------------------------------------------

unmapped_alternatives_df = (
    product_alternatives_df
    .join(
        machine_alternatives_df,
        on="alternative",
        how="left_anti"
    )
)


unmapped_alternative_count = (
    unmapped_alternatives_df.count()
)


# ----------------------------------------------------------------
# 4. Retrieve affected product-alternative relationships
# ----------------------------------------------------------------

affected_product_alternatives_df = (
    bridge_product_production_df
    .join(
        unmapped_alternatives_df,
        on="alternative",
        how="inner"
    )
    .select(
        "product",
        "alternative",
        "main_alternative"
    )
    .orderBy(
        "alternative",
        "product"
    )
)


affected_relationship_count = (
    affected_product_alternatives_df.count()
)

affected_product_count = (
    affected_product_alternatives_df
    .select("product")
    .distinct()
    .count()
)


# ----------------------------------------------------------------
# 5. Check whether affected products exist in dim_product
# ----------------------------------------------------------------

affected_products_df = (
    affected_product_alternatives_df
    .select("product")
    .distinct()
)

affected_products_missing_from_dimension = (
    affected_products_df
    .join(
        dim_product_df.select("product"),
        on="product",
        how="left_anti"
    )
    .count()
)


# ----------------------------------------------------------------
# 6. Profile main_alternative codes for affected relationships
# ----------------------------------------------------------------

affected_main_alternative_distribution_df = (
    affected_product_alternatives_df
    .groupBy("main_alternative")
    .count()
    .orderBy("main_alternative")
)


# ----------------------------------------------------------------
# 7. Display evidence
# ----------------------------------------------------------------

print("UNMAPPED ALTERNATIVES")
print("-" * 120)

display(
    unmapped_alternatives_df
    .orderBy("alternative")
)


print()
print("AFFECTED PRODUCT-ALTERNATIVE RELATIONSHIPS")
print("-" * 120)

display(
    affected_product_alternatives_df
)


print()
print("AFFECTED main_alternative DISTRIBUTION")
print("-" * 120)

display(
    affected_main_alternative_distribution_df
)


# ----------------------------------------------------------------
# 8. Quantify coverage
# ----------------------------------------------------------------

total_product_alternatives = (
    product_alternatives_df.count()
)

mapped_product_alternatives = (
    total_product_alternatives
    - unmapped_alternative_count
)

alternative_machine_coverage_pct = (
    mapped_product_alternatives
    / total_product_alternatives
    * 100
    if total_product_alternatives > 0
    else None
)


# ----------------------------------------------------------------
# 9. Determine investigation status
# ----------------------------------------------------------------

investigation_status = (
    "DOCUMENTED"
    if (
        unmapped_alternative_count == 9
        and affected_products_missing_from_dimension == 0
    )
    else "REVIEW"
)


print()
print("=" * 120)
print("UNMAPPED ALTERNATIVE INVESTIGATION SUMMARY")
print("-" * 120)

print(
    f"Total product-linked alternatives:       "
    f"{total_product_alternatives}"
)

print(
    f"Alternatives with machine relationship:  "
    f"{mapped_product_alternatives}"
)

print(
    f"Alternatives without machine relationship:"
    f" {unmapped_alternative_count}"
)

print(
    f"Alternative-machine coverage:            "
    f"{alternative_machine_coverage_pct:.2f}%"
)

print(
    f"Affected product-alt relationships:      "
    f"{affected_relationship_count}"
)

print(
    f"Distinct affected products:              "
    f"{affected_product_count}"
)

print(
    f"Affected products missing dim_product:   "
    f"{affected_products_missing_from_dimension}"
)

print(
    f"Investigation status:                    "
    f"{investigation_status}"
)

print("=" * 120)


# ----------------------------------------------------------------
# 10. Governance decision
# ----------------------------------------------------------------

if investigation_status == "DOCUMENTED":

    print(
        "GOLD MODELING DECISION: PRESERVE — "
        "the nine product-linked alternatives without machine "
        "relationships will remain in bridge_product_production."
    )

    print(
        "No machine assignment will be inferred or fabricated."
    )

    print(
        "The coverage difference will be documented as an observed "
        "source relationship condition for downstream analysis."
    )

else:

    print(
        "REVIEW REQUIRED — observed alternative coverage differs "
        "from the previously profiled relationship condition."
    )

GOLD DATA WAREHOUSE — UNMAPPED PRODUCTION ALTERNATIVE INVESTIGATION
UNMAPPED ALTERNATIVES
------------------------------------------------------------------------------------------------------------------------


alternative
14251
14255
14262
14269
14307
15279
15460
15463
15464



AFFECTED PRODUCT-ALTERNATIVE RELATIONSHIPS
------------------------------------------------------------------------------------------------------------------------


product,alternative,main_alternative
8000068T,14251,1
8000083T,14255,1
8000105T,14262,1
8000120T,14269,1
8000219T,14307,1
8051071S,15279,1
8081615B,15460,1
8081735B,15463,1
8081736B,15464,1



AFFECTED main_alternative DISTRIBUTION
------------------------------------------------------------------------------------------------------------------------


main_alternative,count
1,9



UNMAPPED ALTERNATIVE INVESTIGATION SUMMARY
------------------------------------------------------------------------------------------------------------------------
Total product-linked alternatives:       592
Alternatives with machine relationship:  583
Alternatives without machine relationship: 9
Alternative-machine coverage:            98.48%
Affected product-alt relationships:      9
Distinct affected products:              9
Affected products missing dim_product:   0
Investigation status:                    DOCUMENTED
GOLD MODELING DECISION: PRESERVE — the nine product-linked alternatives without machine relationships will remain in bridge_product_production.
No machine assignment will be inferred or fabricated.
The coverage difference will be documented as an observed source relationship condition for downstream analysis.


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Gold Data Warehouse — Step 08:
# Pre-Persistence Reconciliation Gate
#
# Purpose:
# Perform one final reconciliation of all 12 proposed Gold
# DataFrames before creating the Gold schema or writing Delta tables.
#
# No Gold Delta tables are persisted in this step.

from pyspark.sql import functions as F

print("GOLD DATA WAREHOUSE — PRE-PERSISTENCE RECONCILIATION GATE")
print("=" * 120)


# ----------------------------------------------------------------
# 1. Register all validated Gold DataFrames
# ----------------------------------------------------------------

gold_dataframes = {

    # Dimensions
    "dim_product":
        dim_product_df,

    "dim_planning_period":
        dim_planning_period_df,

    "dim_machine":
        dim_machine_df,

    "dim_packing_machine":
        dim_packing_machine_df,

    # Facts
    "fact_customer_order":
        fact_customer_order_df,

    "fact_demand_plan":
        fact_demand_plan_df,

    "fact_machine_capacity":
        fact_machine_capacity_df,

    "fact_packing_capacity":
        fact_packing_capacity_df,

    # Bridges
    "bridge_product_production":
        bridge_product_production_df,

    "bridge_alternative_machine":
        bridge_alternative_machine_df,

    "bridge_product_packing":
        bridge_product_packing_df,

    "bridge_product_raw_material":
        bridge_product_raw_material_df
}


# ----------------------------------------------------------------
# 2. Define expected validated row counts
# ----------------------------------------------------------------

expected_gold_rows = {

    "dim_product": 292,
    "dim_planning_period": 8,
    "dim_machine": 39,
    "dim_packing_machine": 7,

    "fact_customer_order": 61,
    "fact_demand_plan": 246,
    "fact_machine_capacity": 312,
    "fact_packing_capacity": 56,

    "bridge_product_production": 592,
    "bridge_alternative_machine": 734,
    "bridge_product_packing": 189,
    "bridge_product_raw_material": 292
}


# ----------------------------------------------------------------
# 3. Reconcile every Gold DataFrame
# ----------------------------------------------------------------

reconciliation_results = []

for table_name, df in gold_dataframes.items():

    actual_rows = df.count()

    expected_rows = expected_gold_rows[
        table_name
    ]

    status = (
        "PASS"
        if actual_rows == expected_rows
        else "FAIL"
    )

    reconciliation_results.append(
        {
            "gold_table": table_name,
            "expected_rows": expected_rows,
            "actual_rows": actual_rows,
            "difference":
                actual_rows - expected_rows,
            "status": status
        }
    )


reconciliation_results_df = (
    spark.createDataFrame(
        reconciliation_results
    )
)


display(
    reconciliation_results_df
    .orderBy("gold_table")
)


# ----------------------------------------------------------------
# 4. Reconcile model composition
# ----------------------------------------------------------------

dimension_tables = [
    name
    for name in gold_dataframes
    if name.startswith("dim_")
]

fact_tables = [
    name
    for name in gold_dataframes
    if name.startswith("fact_")
]

bridge_tables = [
    name
    for name in gold_dataframes
    if name.startswith("bridge_")
]


dimension_rows = sum(
    expected_gold_rows[name]
    for name in dimension_tables
)

fact_rows = sum(
    expected_gold_rows[name]
    for name in fact_tables
)

bridge_rows = sum(
    expected_gold_rows[name]
    for name in bridge_tables
)

expected_total_gold_rows = (
    dimension_rows
    + fact_rows
    + bridge_rows
)

actual_total_gold_rows = sum(
    result["actual_rows"]
    for result in reconciliation_results
)


# ----------------------------------------------------------------
# 5. Reconfirm documented alternative condition
# ----------------------------------------------------------------

unmapped_alternative_count_check = (
    product_alternatives_df
    .join(
        machine_alternatives_df,
        on="alternative",
        how="left_anti"
    )
    .count()
)

alternative_condition_status = (
    "DOCUMENTED"
    if unmapped_alternative_count_check == 9
    else "REVIEW"
)


# ----------------------------------------------------------------
# 6. Reconfirm source-code preservation
# ----------------------------------------------------------------

current_main_alternative_domain = [
    row["main_alternative"]
    for row in (
        bridge_product_production_df
        .select("main_alternative")
        .distinct()
        .orderBy("main_alternative")
        .collect()
    )
]

source_code_status = (
    "PASS"
    if current_main_alternative_domain == [1, 2]
    else "FAIL"
)


# ----------------------------------------------------------------
# 7. Overall reconciliation status
# ----------------------------------------------------------------

failed_tables = sum(
    1
    for result in reconciliation_results
    if result["status"] == "FAIL"
)

table_reconciliation_status = (
    "PASS"
    if failed_tables == 0
    else "FAIL"
)

total_reconciliation_status = (
    "PASS"
    if (
        actual_total_gold_rows
        == expected_total_gold_rows
    )
    else "FAIL"
)


# ----------------------------------------------------------------
# 8. Final summary
# ----------------------------------------------------------------

print()
print("=" * 120)
print("GOLD PRE-PERSISTENCE RECONCILIATION SUMMARY")
print("-" * 120)

print(
    f"Gold DataFrames registered:        "
    f"{len(gold_dataframes)}"
)

print(
    f"Dimensions:                        "
    f"{len(dimension_tables)}"
)

print(
    f"Facts:                             "
    f"{len(fact_tables)}"
)

print(
    f"Bridges:                           "
    f"{len(bridge_tables)}"
)

print()
print("ROW RECONCILIATION")
print("-" * 120)

print(
    f"Dimension rows:                    "
    f"{dimension_rows}"
)

print(
    f"Fact rows:                         "
    f"{fact_rows}"
)

print(
    f"Bridge rows:                       "
    f"{bridge_rows}"
)

print(
    f"Expected total Gold rows:          "
    f"{expected_total_gold_rows}"
)

print(
    f"Actual total Gold rows:            "
    f"{actual_total_gold_rows}"
)

print(
    f"Tables reconciled:                 "
    f"{len(gold_dataframes) - failed_tables}"
    f"/{len(gold_dataframes)}"
)

print(
    f"Table reconciliation:              "
    f"{table_reconciliation_status}"
)

print(
    f"Total-row reconciliation:          "
    f"{total_reconciliation_status}"
)

print()
print("GOVERNANCE CONTROLS")
print("-" * 120)

print(
    f"Unmapped production alternatives:  "
    f"{unmapped_alternative_count_check}"
)

print(
    f"Alternative condition:             "
    f"{alternative_condition_status}"
)

print(
    f"main_alternative domain:           "
    f"{current_main_alternative_domain}"
)

print(
    f"Source-code preservation:          "
    f"{source_code_status}"
)

print("=" * 120)


# ----------------------------------------------------------------
# 9. Persistence gate
# ----------------------------------------------------------------

if (
    table_reconciliation_status == "PASS"
    and total_reconciliation_status == "PASS"
    and alternative_condition_status == "DOCUMENTED"
    and source_code_status == "PASS"
):

    print(
        "GOLD PRE-PERSISTENCE GATE: PASS — "
        "all 12 Gold DataFrames reconcile with their "
        "validated model definitions."
    )

    print(
        "The observed nine unmapped production alternatives "
        "remain documented and preserved."
    )

    print(
        "Gold Delta persistence is authorized for the next step."
    )

else:

    raise ValueError(
        "GOLD PRE-PERSISTENCE GATE FAILED: "
        "do not write Gold Delta tables until all "
        "reconciliation and governance controls pass."
    )

GOLD DATA WAREHOUSE — PRE-PERSISTENCE RECONCILIATION GATE


actual_rows,difference,expected_rows,gold_table,status
734,0,734,bridge_alternative_machine,PASS
189,0,189,bridge_product_packing,PASS
592,0,592,bridge_product_production,PASS
292,0,292,bridge_product_raw_material,PASS
39,0,39,dim_machine,PASS
7,0,7,dim_packing_machine,PASS
8,0,8,dim_planning_period,PASS
292,0,292,dim_product,PASS
61,0,61,fact_customer_order,PASS
246,0,246,fact_demand_plan,PASS



GOLD PRE-PERSISTENCE RECONCILIATION SUMMARY
------------------------------------------------------------------------------------------------------------------------
Gold DataFrames registered:        12
Dimensions:                        4
Facts:                             4
Bridges:                           4

ROW RECONCILIATION
------------------------------------------------------------------------------------------------------------------------
Dimension rows:                    346
Fact rows:                         675
Bridge rows:                       1807
Expected total Gold rows:          2828
Actual total Gold rows:            2828
Tables reconciled:                 12/12
Table reconciliation:              PASS
Total-row reconciliation:          PASS

GOVERNANCE CONTROLS
------------------------------------------------------------------------------------------------------------------------
Unmapped production alternatives:  9
Alternative condition:             DOCUMENTED


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Gold Data Warehouse — Step 09:
# Persist Validated Gold Delta Tables
#
# Purpose:
# Create the Gold schema and persist the 12 validated Gold
# DataFrames as managed Delta tables.
#
# Persistence strategy:
# - CREATE SCHEMA IF NOT EXISTS
# - overwrite each Gold analytical table
# - preserve the validated model exactly as constructed
# - reconcile persisted row counts immediately after writing

from pyspark.sql import functions as F

print("GOLD DATA WAREHOUSE — PERSIST GOLD DELTA TABLES")
print("=" * 120)

gold_catalog = "workspace"
gold_schema = "gold_sop"

gold_prefix = f"{gold_catalog}.{gold_schema}"


# ----------------------------------------------------------------
# 1. Reconfirm the 12 validated Gold DataFrames are available
# ----------------------------------------------------------------

required_gold_dataframes = {
    "dim_product": dim_product_df,
    "dim_planning_period": dim_planning_period_df,
    "dim_machine": dim_machine_df,
    "dim_packing_machine": dim_packing_machine_df,

    "fact_customer_order": fact_customer_order_df,
    "fact_demand_plan": fact_demand_plan_df,
    "fact_machine_capacity": fact_machine_capacity_df,
    "fact_packing_capacity": fact_packing_capacity_df,

    "bridge_product_production": bridge_product_production_df,
    "bridge_alternative_machine": bridge_alternative_machine_df,
    "bridge_product_packing": bridge_product_packing_df,
    "bridge_product_raw_material": bridge_product_raw_material_df
}


expected_gold_rows = {
    "dim_product": 292,
    "dim_planning_period": 8,
    "dim_machine": 39,
    "dim_packing_machine": 7,

    "fact_customer_order": 61,
    "fact_demand_plan": 246,
    "fact_machine_capacity": 312,
    "fact_packing_capacity": 56,

    "bridge_product_production": 592,
    "bridge_alternative_machine": 734,
    "bridge_product_packing": 189,
    "bridge_product_raw_material": 292
}


# ----------------------------------------------------------------
# 2. Pre-write safety check
# ----------------------------------------------------------------

prewrite_results = []

for table_name, df in required_gold_dataframes.items():

    actual_rows = df.count()
    expected_rows = expected_gold_rows[table_name]

    status = (
        "PASS"
        if actual_rows == expected_rows
        else "FAIL"
    )

    prewrite_results.append(
        {
            "gold_table": table_name,
            "expected_rows": expected_rows,
            "actual_rows": actual_rows,
            "status": status
        }
    )


prewrite_failures = sum(
    1
    for result in prewrite_results
    if result["status"] == "FAIL"
)


if prewrite_failures > 0:

    raise ValueError(
        "PRE-WRITE SAFETY CHECK FAILED: "
        "one or more Gold DataFrames no longer reconcile "
        "with the validated row counts."
    )


print(
    "Pre-write safety check:        "
    "PASS — 12/12 Gold DataFrames reconcile."
)


# ----------------------------------------------------------------
# 3. Create Gold schema
# ----------------------------------------------------------------

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{gold_catalog}.{gold_schema}"
)

print(
    f"Gold schema ready:             "
    f"{gold_prefix}"
)


# ----------------------------------------------------------------
# 4. Persist Gold DataFrames as managed Delta tables
# ----------------------------------------------------------------

write_results = []

for table_name, df in required_gold_dataframes.items():

    target_table = (
        f"{gold_prefix}.{table_name}"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(target_table)
    )

    persisted_rows = (
        spark.table(target_table).count()
    )

    expected_rows = (
        expected_gold_rows[table_name]
    )

    status = (
        "PASS"
        if persisted_rows == expected_rows
        else "FAIL"
    )

    write_results.append(
        {
            "gold_table": table_name,
            "target_table": target_table,
            "expected_rows": expected_rows,
            "persisted_rows": persisted_rows,
            "status": status
        }
    )


# ----------------------------------------------------------------
# 5. Display persistence results
# ----------------------------------------------------------------

write_results_df = (
    spark.createDataFrame(
        write_results
    )
)

display(
    write_results_df
    .orderBy("gold_table")
)


# ----------------------------------------------------------------
# 6. Verify physical Gold table inventory
# ----------------------------------------------------------------

persisted_gold_tables = sorted(
    [
        row["tableName"]
        for row in (
            spark.sql(
                f"SHOW TABLES IN "
                f"{gold_prefix}"
            )
            .select("tableName")
            .collect()
        )
    ]
)

expected_gold_tables = sorted(
    required_gold_dataframes.keys()
)

missing_gold_tables = sorted(
    set(expected_gold_tables)
    - set(persisted_gold_tables)
)

unexpected_gold_tables = sorted(
    set(persisted_gold_tables)
    - set(expected_gold_tables)
)


# ----------------------------------------------------------------
# 7. Reconcile total persisted records
# ----------------------------------------------------------------

expected_total_rows = sum(
    expected_gold_rows.values()
)

persisted_total_rows = sum(
    result["persisted_rows"]
    for result in write_results
)

failed_writes = sum(
    1
    for result in write_results
    if result["status"] == "FAIL"
)


table_inventory_status = (
    "PASS"
    if (
        len(missing_gold_tables) == 0
        and len(unexpected_gold_tables) == 0
        and len(persisted_gold_tables) == 12
    )
    else "FAIL"
)

row_reconciliation_status = (
    "PASS"
    if (
        failed_writes == 0
        and persisted_total_rows == expected_total_rows
    )
    else "FAIL"
)


# ----------------------------------------------------------------
# 8. Persistence summary
# ----------------------------------------------------------------

print()
print("=" * 120)
print("GOLD DELTA PERSISTENCE SUMMARY")
print("-" * 120)

print(
    f"Gold target schema:              "
    f"{gold_prefix}"
)

print(
    f"Expected Gold tables:            "
    f"{len(expected_gold_tables)}"
)

print(
    f"Persisted Gold tables:           "
    f"{len(persisted_gold_tables)}"
)

print(
    f"Missing Gold tables:             "
    f"{len(missing_gold_tables)}"
)

print(
    f"Unexpected Gold tables:          "
    f"{len(unexpected_gold_tables)}"
)

print(
    f"Table inventory validation:      "
    f"{table_inventory_status}"
)

print()
print("ROW PERSISTENCE")
print("-" * 120)

print(
    f"Expected Gold records:           "
    f"{expected_total_rows}"
)

print(
    f"Persisted Gold records:          "
    f"{persisted_total_rows}"
)

print(
    f"Tables with row-count failure:   "
    f"{failed_writes}"
)

print(
    f"Row reconciliation:              "
    f"{row_reconciliation_status}"
)

print("=" * 120)


# ----------------------------------------------------------------
# 9. Final persistence control
# ----------------------------------------------------------------

if (
    table_inventory_status == "PASS"
    and row_reconciliation_status == "PASS"
):

    print(
        "GOLD DELTA PERSISTENCE STATUS: PASS — "
        "all 12 validated Gold analytical tables were "
        "successfully persisted as managed Delta tables."
    )

    print(
        "Gold persistence is complete. "
        "Post-persistence semantic validation is the next step."
    )

else:

    raise ValueError(
        "GOLD DELTA PERSISTENCE FAILED: "
        "one or more Gold table inventory or row-count "
        "controls did not reconcile."
    )

GOLD DATA WAREHOUSE — PERSIST GOLD DELTA TABLES
Pre-write safety check:        PASS — 12/12 Gold DataFrames reconcile.
Gold schema ready:             workspace.gold_sop


expected_rows,gold_table,persisted_rows,status,target_table
734,bridge_alternative_machine,734,PASS,workspace.gold_sop.bridge_alternative_machine
189,bridge_product_packing,189,PASS,workspace.gold_sop.bridge_product_packing
592,bridge_product_production,592,PASS,workspace.gold_sop.bridge_product_production
292,bridge_product_raw_material,292,PASS,workspace.gold_sop.bridge_product_raw_material
39,dim_machine,39,PASS,workspace.gold_sop.dim_machine
7,dim_packing_machine,7,PASS,workspace.gold_sop.dim_packing_machine
8,dim_planning_period,8,PASS,workspace.gold_sop.dim_planning_period
292,dim_product,292,PASS,workspace.gold_sop.dim_product
61,fact_customer_order,61,PASS,workspace.gold_sop.fact_customer_order
246,fact_demand_plan,246,PASS,workspace.gold_sop.fact_demand_plan



GOLD DELTA PERSISTENCE SUMMARY
------------------------------------------------------------------------------------------------------------------------
Gold target schema:              workspace.gold_sop
Expected Gold tables:            12
Persisted Gold tables:           12
Missing Gold tables:             0
Unexpected Gold tables:          0
Table inventory validation:      PASS

ROW PERSISTENCE
------------------------------------------------------------------------------------------------------------------------
Expected Gold records:           2828
Persisted Gold records:          2828
Tables with row-count failure:   0
Row reconciliation:              PASS
GOLD DELTA PERSISTENCE STATUS: PASS — all 12 validated Gold analytical tables were successfully persisted as managed Delta tables.
Gold persistence is complete. Post-persistence semantic validation is the next step.


In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Gold Data Warehouse — Step 10:
# Post-Persistence Semantic Validation
#
# Purpose:
# Validate the persisted Gold Delta tables directly from
# workspace.gold_sop rather than relying on in-memory DataFrames.
#
# Controls:
# - persisted table inventory
# - primary analytical grain
# - fact-to-dimension relationships
# - bridge-to-dimension relationships
# - planning-horizon structure
# - customer-order horizon classification
# - source-code preservation
# - documented unmapped production alternatives
#
# No data is modified in this step.

from pyspark.sql import functions as F

print("GOLD DATA WAREHOUSE — POST-PERSISTENCE SEMANTIC VALIDATION")
print("=" * 120)

gold_catalog = "workspace"
gold_schema = "gold_sop"

gold_prefix = f"{gold_catalog}.{gold_schema}"


# ----------------------------------------------------------------
# 1. Load persisted Gold Delta tables
# ----------------------------------------------------------------

dim_product_gold = spark.table(
    f"{gold_prefix}.dim_product"
)

dim_planning_period_gold = spark.table(
    f"{gold_prefix}.dim_planning_period"
)

dim_machine_gold = spark.table(
    f"{gold_prefix}.dim_machine"
)

dim_packing_machine_gold = spark.table(
    f"{gold_prefix}.dim_packing_machine"
)

fact_customer_order_gold = spark.table(
    f"{gold_prefix}.fact_customer_order"
)

fact_demand_plan_gold = spark.table(
    f"{gold_prefix}.fact_demand_plan"
)

fact_machine_capacity_gold = spark.table(
    f"{gold_prefix}.fact_machine_capacity"
)

fact_packing_capacity_gold = spark.table(
    f"{gold_prefix}.fact_packing_capacity"
)

bridge_product_production_gold = spark.table(
    f"{gold_prefix}.bridge_product_production"
)

bridge_alternative_machine_gold = spark.table(
    f"{gold_prefix}.bridge_alternative_machine"
)

bridge_product_packing_gold = spark.table(
    f"{gold_prefix}.bridge_product_packing"
)

bridge_product_raw_material_gold = spark.table(
    f"{gold_prefix}.bridge_product_raw_material"
)


# ----------------------------------------------------------------
# 2. Persisted Gold grain controls
# ----------------------------------------------------------------

grain_tests = [

    (
        "dim_product",
        dim_product_gold,
        ["product"],
        292
    ),

    (
        "dim_planning_period",
        dim_planning_period_gold,
        ["period"],
        8
    ),

    (
        "dim_machine",
        dim_machine_gold,
        ["machine"],
        39
    ),

    (
        "dim_packing_machine",
        dim_packing_machine_gold,
        ["packing_machine"],
        7
    ),

    (
        "fact_customer_order",
        fact_customer_order_gold,
        ["order_id"],
        61
    ),

    (
        "fact_demand_plan",
        fact_demand_plan_gold,
        ["product", "period"],
        246
    ),

    (
        "fact_machine_capacity",
        fact_machine_capacity_gold,
        ["machine", "period"],
        312
    ),

    (
        "fact_packing_capacity",
        fact_packing_capacity_gold,
        ["packing_machine", "period"],
        56
    ),

    (
        "bridge_product_production",
        bridge_product_production_gold,
        ["product", "alternative"],
        592
    ),

    (
        "bridge_alternative_machine",
        bridge_alternative_machine_gold,
        ["alternative", "machine"],
        734
    ),

    (
        "bridge_product_packing",
        bridge_product_packing_gold,
        ["product", "packing_machine"],
        189
    ),

    (
        "bridge_product_raw_material",
        bridge_product_raw_material_gold,
        ["product"],
        292
    )
]


grain_results = []

for (
    table_name,
    df,
    key_columns,
    expected_rows
) in grain_tests:

    actual_rows = df.count()

    distinct_keys = (
        df
        .select(*key_columns)
        .distinct()
        .count()
    )

    duplicate_key_rows = (
        actual_rows - distinct_keys
    )

    status = (
        "PASS"
        if (
            actual_rows == expected_rows
            and distinct_keys == actual_rows
            and duplicate_key_rows == 0
        )
        else "FAIL"
    )

    grain_results.append(
        {
            "gold_table": table_name,
            "expected_rows": expected_rows,
            "actual_rows": actual_rows,
            "distinct_keys": distinct_keys,
            "duplicate_key_rows": duplicate_key_rows,
            "status": status
        }
    )


grain_results_df = spark.createDataFrame(
    grain_results
)

display(
    grain_results_df
    .orderBy("gold_table")
)


# ----------------------------------------------------------------
# 3. Relationship-validation helper
# ----------------------------------------------------------------

def persisted_orphan_count(
    child_df,
    child_column,
    parent_df,
    parent_column
):

    child_keys = (
        child_df
        .select(
            F.col(child_column).alias("join_key")
        )
        .filter(
            F.col("join_key").isNotNull()
        )
        .distinct()
    )

    parent_keys = (
        parent_df
        .select(
            F.col(parent_column).alias("join_key")
        )
        .filter(
            F.col("join_key").isNotNull()
        )
        .distinct()
    )

    return (
        child_keys
        .join(
            parent_keys,
            on="join_key",
            how="left_anti"
        )
        .count()
    )


# ----------------------------------------------------------------
# 4. Fact → dimension relationships
# ----------------------------------------------------------------

fact_relationships = [

    (
        "fact_customer_order",
        fact_customer_order_gold,
        "product",
        "dim_product",
        dim_product_gold,
        "product"
    ),

    (
        "fact_demand_plan",
        fact_demand_plan_gold,
        "product",
        "dim_product",
        dim_product_gold,
        "product"
    ),

    (
        "fact_demand_plan",
        fact_demand_plan_gold,
        "period",
        "dim_planning_period",
        dim_planning_period_gold,
        "period"
    ),

    (
        "fact_machine_capacity",
        fact_machine_capacity_gold,
        "machine",
        "dim_machine",
        dim_machine_gold,
        "machine"
    ),

    (
        "fact_machine_capacity",
        fact_machine_capacity_gold,
        "period",
        "dim_planning_period",
        dim_planning_period_gold,
        "period"
    ),

    (
        "fact_packing_capacity",
        fact_packing_capacity_gold,
        "packing_machine",
        "dim_packing_machine",
        dim_packing_machine_gold,
        "packing_machine"
    ),

    (
        "fact_packing_capacity",
        fact_packing_capacity_gold,
        "period",
        "dim_planning_period",
        dim_planning_period_gold,
        "period"
    )
]


fact_relationship_results = []

for (
    child_name,
    child_df,
    child_column,
    parent_name,
    parent_df,
    parent_column
) in fact_relationships:

    orphan_keys = persisted_orphan_count(
        child_df,
        child_column,
        parent_df,
        parent_column
    )

    fact_relationship_results.append(
        {
            "child_table": child_name,
            "child_key": child_column,
            "parent_table": parent_name,
            "parent_key": parent_column,
            "orphan_keys": orphan_keys,
            "status": (
                "PASS"
                if orphan_keys == 0
                else "FAIL"
            )
        }
    )


# ----------------------------------------------------------------
# 5. Bridge → dimension relationships
# ----------------------------------------------------------------

bridge_relationships = [

    (
        "bridge_product_production",
        bridge_product_production_gold,
        "product",
        "dim_product",
        dim_product_gold,
        "product"
    ),

    (
        "bridge_alternative_machine",
        bridge_alternative_machine_gold,
        "machine",
        "dim_machine",
        dim_machine_gold,
        "machine"
    ),

    (
        "bridge_product_packing",
        bridge_product_packing_gold,
        "product",
        "dim_product",
        dim_product_gold,
        "product"
    ),

    (
        "bridge_product_packing",
        bridge_product_packing_gold,
        "packing_machine",
        "dim_packing_machine",
        dim_packing_machine_gold,
        "packing_machine"
    ),

    (
        "bridge_product_raw_material",
        bridge_product_raw_material_gold,
        "product",
        "dim_product",
        dim_product_gold,
        "product"
    )
]


bridge_relationship_results = []

for (
    child_name,
    child_df,
    child_column,
    parent_name,
    parent_df,
    parent_column
) in bridge_relationships:

    orphan_keys = persisted_orphan_count(
        child_df,
        child_column,
        parent_df,
        parent_column
    )

    bridge_relationship_results.append(
        {
            "child_table": child_name,
            "child_key": child_column,
            "parent_table": parent_name,
            "parent_key": parent_column,
            "orphan_keys": orphan_keys,
            "status": (
                "PASS"
                if orphan_keys == 0
                else "FAIL"
            )
        }
    )


all_relationship_results = (
    fact_relationship_results
    + bridge_relationship_results
)

relationship_results_df = (
    spark.createDataFrame(
        all_relationship_results
    )
)

print()
print("PERSISTED GOLD RELATIONSHIP VALIDATION")
print("-" * 120)

display(
    relationship_results_df
    .orderBy(
        "child_table",
        "parent_table"
    )
)


# ----------------------------------------------------------------
# 6. Planning-period semantic control
# ----------------------------------------------------------------

period_count = (
    dim_planning_period_gold.count()
)

period_min = (
    dim_planning_period_gold
    .agg(F.min("period"))
    .first()[0]
)

period_max = (
    dim_planning_period_gold
    .agg(F.max("period"))
    .first()[0]
)

total_horizon_days = (
    dim_planning_period_gold
    .agg(F.sum("number_of_days"))
    .first()[0]
)

final_horizon_day = (
    dim_planning_period_gold
    .agg(F.max("last_day_of_period"))
    .first()[0]
)

planning_period_status = (
    "PASS"
    if (
        period_count == 8
        and period_min == 1
        and period_max == 8
        and total_horizon_days == 101
        and final_horizon_day == 101
    )
    else "FAIL"
)


# ----------------------------------------------------------------
# 7. Customer-order horizon semantic control
# ----------------------------------------------------------------

before_horizon = (
    fact_customer_order_gold
    .filter(
        F.col("planning_horizon_status")
        == "Before horizon start"
    )
    .count()
)

within_horizon = (
    fact_customer_order_gold
    .filter(
        F.col("planning_horizon_status")
        == "Within horizon"
    )
    .count()
)

beyond_horizon = (
    fact_customer_order_gold
    .filter(
        F.col("planning_horizon_status")
        == "Beyond horizon end"
    )
    .count()
)

customer_horizon_status = (
    "PASS"
    if (
        before_horizon == 9
        and within_horizon == 52
        and beyond_horizon == 0
    )
    else "FAIL"
)


# ----------------------------------------------------------------
# 8. Source-code preservation
# ----------------------------------------------------------------

main_alternative_domain = [
    row["main_alternative"]
    for row in (
        bridge_product_production_gold
        .select("main_alternative")
        .distinct()
        .orderBy("main_alternative")
        .collect()
    )
]

source_code_status = (
    "PASS"
    if main_alternative_domain == [1, 2]
    else "FAIL"
)


# ----------------------------------------------------------------
# 9. Revalidate documented unmapped alternative condition
# ----------------------------------------------------------------

persisted_product_alternatives = (
    bridge_product_production_gold
    .select("alternative")
    .distinct()
)

persisted_machine_alternatives = (
    bridge_alternative_machine_gold
    .select("alternative")
    .distinct()
)

persisted_unmapped_alternatives = (
    persisted_product_alternatives
    .join(
        persisted_machine_alternatives,
        on="alternative",
        how="left_anti"
    )
    .count()
)

alternative_condition_status = (
    "DOCUMENTED"
    if persisted_unmapped_alternatives == 9
    else "REVIEW"
)


# ----------------------------------------------------------------
# 10. Overall validation results
# ----------------------------------------------------------------

failed_grain_tests = sum(
    1
    for result in grain_results
    if result["status"] == "FAIL"
)

failed_relationship_tests = sum(
    1
    for result in all_relationship_results
    if result["status"] == "FAIL"
)

grain_status = (
    "PASS"
    if failed_grain_tests == 0
    else "FAIL"
)

relationship_status = (
    "PASS"
    if failed_relationship_tests == 0
    else "FAIL"
)


# ----------------------------------------------------------------
# 11. Final semantic validation summary
# ----------------------------------------------------------------

print()
print("=" * 120)
print("GOLD POST-PERSISTENCE SEMANTIC VALIDATION SUMMARY")
print("-" * 120)

print(
    f"Persisted Gold tables tested:       "
    f"{len(grain_results)}"
)

print(
    f"Grain tests passed:                 "
    f"{len(grain_results) - failed_grain_tests}"
    f"/{len(grain_results)}"
)

print(
    f"Grain validation:                   "
    f"{grain_status}"
)

print(
    f"Relationships tested:               "
    f"{len(all_relationship_results)}"
)

print(
    f"Relationships passed:               "
    f"{len(all_relationship_results) - failed_relationship_tests}"
    f"/{len(all_relationship_results)}"
)

print(
    f"Relationship validation:            "
    f"{relationship_status}"
)

print()
print("SEMANTIC CONTROLS")
print("-" * 120)

print(
    f"Planning-period structure:          "
    f"{planning_period_status}"
)

print(
    f"Customer-order horizon control:     "
    f"{customer_horizon_status}"
)

print(
    f"Before / Within / Beyond horizon:   "
    f"{before_horizon} / "
    f"{within_horizon} / "
    f"{beyond_horizon}"
)

print(
    f"main_alternative domain:            "
    f"{main_alternative_domain}"
)

print(
    f"Source-code preservation:           "
    f"{source_code_status}"
)

print(
    f"Unmapped production alternatives:   "
    f"{persisted_unmapped_alternatives}"
)

print(
    f"Alternative condition:              "
    f"{alternative_condition_status}"
)

print("=" * 120)


# ----------------------------------------------------------------
# 12. Final acceptance gate
# ----------------------------------------------------------------

if (
    grain_status == "PASS"
    and relationship_status == "PASS"
    and planning_period_status == "PASS"
    and customer_horizon_status == "PASS"
    and source_code_status == "PASS"
    and alternative_condition_status == "DOCUMENTED"
):

    print(
        "GOLD POST-PERSISTENCE VALIDATION STATUS: PASS — "
        "the persisted Gold analytical model preserves its "
        "validated grain, relationships, and source-supported semantics."
    )

    print(
        "The Gold Data Warehouse is ready for final completion "
        "documentation."
    )

else:

    raise ValueError(
        "GOLD POST-PERSISTENCE VALIDATION FAILED: "
        "the persisted analytical model does not fully reconcile "
        "with its validated Gold design."
    )

GOLD DATA WAREHOUSE — POST-PERSISTENCE SEMANTIC VALIDATION


actual_rows,distinct_keys,duplicate_key_rows,expected_rows,gold_table,status
734,734,0,734,bridge_alternative_machine,PASS
189,189,0,189,bridge_product_packing,PASS
592,592,0,592,bridge_product_production,PASS
292,292,0,292,bridge_product_raw_material,PASS
39,39,0,39,dim_machine,PASS
7,7,0,7,dim_packing_machine,PASS
8,8,0,8,dim_planning_period,PASS
292,292,0,292,dim_product,PASS
61,61,0,61,fact_customer_order,PASS
246,246,0,246,fact_demand_plan,PASS



PERSISTED GOLD RELATIONSHIP VALIDATION
------------------------------------------------------------------------------------------------------------------------


child_key,child_table,orphan_keys,parent_key,parent_table,status
machine,bridge_alternative_machine,0,machine,dim_machine,PASS
packing_machine,bridge_product_packing,0,packing_machine,dim_packing_machine,PASS
product,bridge_product_packing,0,product,dim_product,PASS
product,bridge_product_production,0,product,dim_product,PASS
product,bridge_product_raw_material,0,product,dim_product,PASS
product,fact_customer_order,0,product,dim_product,PASS
period,fact_demand_plan,0,period,dim_planning_period,PASS
product,fact_demand_plan,0,product,dim_product,PASS
machine,fact_machine_capacity,0,machine,dim_machine,PASS
period,fact_machine_capacity,0,period,dim_planning_period,PASS



GOLD POST-PERSISTENCE SEMANTIC VALIDATION SUMMARY
------------------------------------------------------------------------------------------------------------------------
Persisted Gold tables tested:       12
Grain tests passed:                 12/12
Grain validation:                   PASS
Relationships tested:               12
Relationships passed:               12/12
Relationship validation:            PASS

SEMANTIC CONTROLS
------------------------------------------------------------------------------------------------------------------------
Planning-period structure:          PASS
Customer-order horizon control:     PASS
Before / Within / Beyond horizon:   9 / 52 / 0
main_alternative domain:            [1, 2]
Source-code preservation:           PASS
Unmapped production alternatives:   9
Alternative condition:              DOCUMENTED
GOLD POST-PERSISTENCE VALIDATION STATUS: PASS — the persisted Gold analytical model preserves its validated grain, relationships, and source-suppo

In [0]:
# Project #4 — Supply Chain Planning & Operations Intelligence
# Gold Data Warehouse — Step 11:
# Gold Data Warehouse Completion Summary
#
# Purpose:
# Produce a concise final audit summary of the completed
# and persisted Gold analytical layer.
#
# This step reads persisted Gold tables for final reconciliation.
# No data is modified.

from pyspark.sql import functions as F

print("PROJECT #4 — GOLD DATA WAREHOUSE COMPLETION SUMMARY")
print("=" * 100)

gold_catalog = "workspace"
gold_schema = "gold_sop"
gold_prefix = f"{gold_catalog}.{gold_schema}"


# ----------------------------------------------------------------
# 1. Verify persisted Gold inventory
# ----------------------------------------------------------------

expected_gold_tables = [
    "dim_product",
    "dim_planning_period",
    "dim_machine",
    "dim_packing_machine",

    "fact_customer_order",
    "fact_demand_plan",
    "fact_machine_capacity",
    "fact_packing_capacity",

    "bridge_product_production",
    "bridge_alternative_machine",
    "bridge_product_packing",
    "bridge_product_raw_material"
]


persisted_gold_tables = sorted(
    [
        row["tableName"]
        for row in (
            spark.sql(
                f"SHOW TABLES IN {gold_prefix}"
            )
            .select("tableName")
            .collect()
        )
    ]
)


validated_table_count = len(
    set(expected_gold_tables)
    .intersection(
        set(persisted_gold_tables)
    )
)


# ----------------------------------------------------------------
# 2. Recalculate persisted row totals
# ----------------------------------------------------------------

gold_row_counts = {}

for table_name in expected_gold_tables:

    gold_row_counts[table_name] = (
        spark.table(
            f"{gold_prefix}.{table_name}"
        )
        .count()
    )


dimension_tables = [
    name
    for name in expected_gold_tables
    if name.startswith("dim_")
]

fact_tables = [
    name
    for name in expected_gold_tables
    if name.startswith("fact_")
]

bridge_tables = [
    name
    for name in expected_gold_tables
    if name.startswith("bridge_")
]


dimension_rows = sum(
    gold_row_counts[name]
    for name in dimension_tables
)

fact_rows = sum(
    gold_row_counts[name]
    for name in fact_tables
)

bridge_rows = sum(
    gold_row_counts[name]
    for name in bridge_tables
)

total_gold_rows = (
    dimension_rows
    + fact_rows
    + bridge_rows
)


# ----------------------------------------------------------------
# 3. Reconfirm key semantic controls
# ----------------------------------------------------------------

customer_orders = spark.table(
    f"{gold_prefix}.fact_customer_order"
)

product_production = spark.table(
    f"{gold_prefix}.bridge_product_production"
)

alternative_machine = spark.table(
    f"{gold_prefix}.bridge_alternative_machine"
)


before_horizon = (
    customer_orders
    .filter(
        F.col("planning_horizon_status")
        == "Before horizon start"
    )
    .count()
)

within_horizon = (
    customer_orders
    .filter(
        F.col("planning_horizon_status")
        == "Within horizon"
    )
    .count()
)

beyond_horizon = (
    customer_orders
    .filter(
        F.col("planning_horizon_status")
        == "Beyond horizon end"
    )
    .count()
)


main_alternative_domain = [
    row["main_alternative"]
    for row in (
        product_production
        .select("main_alternative")
        .distinct()
        .orderBy("main_alternative")
        .collect()
    )
]


product_alternatives = (
    product_production
    .select("alternative")
    .distinct()
)

machine_alternatives = (
    alternative_machine
    .select("alternative")
    .distinct()
)

unmapped_alternatives = (
    product_alternatives
    .join(
        machine_alternatives,
        on="alternative",
        how="left_anti"
    )
    .count()
)


# ----------------------------------------------------------------
# 4. Final completion controls
# ----------------------------------------------------------------

table_inventory_status = (
    "PASS"
    if (
        validated_table_count == 12
        and len(persisted_gold_tables) == 12
    )
    else "FAIL"
)

row_reconciliation_status = (
    "PASS"
    if total_gold_rows == 2828
    else "FAIL"
)

horizon_control_status = (
    "PASS"
    if (
        before_horizon == 9
        and within_horizon == 52
        and beyond_horizon == 0
    )
    else "FAIL"
)

source_code_status = (
    "PASS"
    if main_alternative_domain == [1, 2]
    else "FAIL"
)

alternative_condition_status = (
    "DOCUMENTED"
    if unmapped_alternatives == 9
    else "REVIEW"
)


# ----------------------------------------------------------------
# 5. Print final Gold completion summary
# ----------------------------------------------------------------

print(f"Gold target schema:          {gold_prefix}")
print(f"Gold analytical tables:      {len(persisted_gold_tables)}")
print(f"Dimensions:                  {len(dimension_tables)}")
print(f"Facts:                       {len(fact_tables)}")
print(f"Bridges:                     {len(bridge_tables)}")
print(f"Persisted Gold records:      {total_gold_rows}")

print()
print("ANALYTICAL MODEL")
print("-" * 100)

print(f"Dimension records:           {dimension_rows}")
print(f"Fact records:                {fact_rows}")
print(f"Bridge records:              {bridge_rows}")
print(f"Validated Gold tables:       {validated_table_count}/12")

print()
print("SEMANTIC VALIDATION")
print("-" * 100)

print("Persisted grain tests:       12/12 PASS")
print("Persisted relationships:     12/12 PASS")
print(f"Customer-order horizon:      {before_horizon} before / "
      f"{within_horizon} within / "
      f"{beyond_horizon} beyond")
print(f"main_alternative domain:     {main_alternative_domain}")
print(f"Unmapped alternatives:       {unmapped_alternatives}")
print(f"Alternative condition:       {alternative_condition_status}")

print()
print("FINAL CONTROLS")
print("-" * 100)

print(f"Table inventory:             {table_inventory_status}")
print(f"Row reconciliation:          {row_reconciliation_status}")
print(f"Horizon classification:      {horizon_control_status}")
print(f"Source-code preservation:    {source_code_status}")

print("=" * 100)


# ----------------------------------------------------------------
# 6. Final Gold acceptance gate
# ----------------------------------------------------------------

if (
    table_inventory_status == "PASS"
    and row_reconciliation_status == "PASS"
    and horizon_control_status == "PASS"
    and source_code_status == "PASS"
    and alternative_condition_status == "DOCUMENTED"
):

    print(
        "GOLD DATA WAREHOUSE STATUS: COMPLETE"
    )

    print(
        "Twelve validated Gold Delta tables are persisted in "
        "workspace.gold_sop and are ready for downstream "
        "SQL, Power BI, and Python analytics."
    )

    print(
        "The nine product-linked production alternatives "
        "without machine relationships remain preserved "
        "and explicitly documented."
    )

else:

    raise ValueError(
        "GOLD DATA WAREHOUSE COMPLETION CHECK FAILED: "
        "one or more final Gold controls did not pass."
    )

PROJECT #4 — GOLD DATA WAREHOUSE COMPLETION SUMMARY
Gold target schema:          workspace.gold_sop
Gold analytical tables:      12
Dimensions:                  4
Facts:                       4
Bridges:                     4
Persisted Gold records:      2828

ANALYTICAL MODEL
----------------------------------------------------------------------------------------------------
Dimension records:           346
Fact records:                675
Bridge records:              1807
Validated Gold tables:       12/12

SEMANTIC VALIDATION
----------------------------------------------------------------------------------------------------
Persisted grain tests:       12/12 PASS
Persisted relationships:     12/12 PASS
Customer-order horizon:      9 before / 52 within / 0 beyond
main_alternative domain:     [1, 2]
Unmapped alternatives:       9
Alternative condition:       DOCUMENTED

FINAL CONTROLS
----------------------------------------------------------------------------------------------------
